In [1]:
# Cell 1 — Imports and configuration
from pathlib import Path
import os
import ultralytics
import sklearn
import shutil
import yaml
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import torch
import json
import math
import warnings


warnings.filterwarnings("ignore")

# ==========================================================
# CORRECT DATASET PATH
# This means: use the folder where this notebook is running
# ==========================================================
DATASET_ROOT = Path.cwd()

SOURCE_IMAGE_DIR = DATASET_ROOT / "train" / "images"
SOURCE_LABEL_DIR = DATASET_ROOT / "train" / "labels"

TREE_MAPPING_CSV = DATASET_ROOT / "image_tree_mapping.csv"
TREE_GPS_REFERENCE_CSV = DATASET_ROOT / "tree_gps_reference.csv"


# Dynamic mapping setup of orchard and trees
N_ORCHARDS = 5
BLOCKS_PER_ORCHARD = 3
ROWS_PER_BLOCK = 4

# Keep True while you are experimenting.
# Later, if you manually edit GPS/tree mapping, change this to False.
RECREATE_TREE_MAPPING_EACH_RUN = True

# Working folder where train/val split will be created
WORK_DIR = DATASET_ROOT / "yolo_research_working"

# Output folder
OUTPUT_DIR = DATASET_ROOT / "research_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Class mapping - verify exact order from dataset- refer classes.txt

CLASS_NAMES = {
    0: "Green",
    1: "Half",
    2: "Red",
    3: "Young"
}
SEED = 42
VAL_SIZE = 0.20
# Training configuration
# Use EPOCHS = 2 only for quick testing.
# Use 50 or 100 for serious model training.
EPOCHS = 50
IMAGE_SIZE = 640

# MacBook Air may struggle with batch=16.
# Use 4 or 8 for safer training.
BATCH_SIZE = 4

BASE_MODEL = "yolo11n.pt"

# NMS / validation safety settings
YOLO_MAX_DET = 100       # reduce from default 300
YOLO_NMS_IOU = 0.50      # reduce from default 0.70
VAL_DEVICE = "cpu"       # use CPU for validation if MPS gives NMS timeout

# Device selection
# For Mac M1/M2/M3, MPS can cause slow NMS during YOLO validation.
# CPU is slower but more stable for validation/NMS.

FORCE_CPU = True

if FORCE_CPU:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = 0
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

VAL_DEVICE = "cpu"

print("Training device:", DEVICE)
print("Validation device:", VAL_DEVICE)



# Save environment details for reproducibility


environment_info = {
    "python_version": os.sys.version,
    "torch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "opencv_version": cv2.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
    "device": str(DEVICE)
}

environment_info_path = OUTPUT_DIR / "environment_info.json"

with open(environment_info_path, "w") as f:
    json.dump(environment_info, f, indent=4)

print("Saved environment info:", environment_info_path)
print("Training device:", DEVICE)
print("Current notebook folder:", DATASET_ROOT)
print("Image folder:", SOURCE_IMAGE_DIR)
print("Label folder:", SOURCE_LABEL_DIR)

print("Image folder exists:", SOURCE_IMAGE_DIR.exists())
print("Label folder exists:", SOURCE_LABEL_DIR.exists())

print("Number of JPG images:", len(list(SOURCE_IMAGE_DIR.glob('*.jpg'))))
print("Number of TXT labels:", len(list(SOURCE_LABEL_DIR.glob('*.txt'))))

if not SOURCE_IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {SOURCE_IMAGE_DIR}")

if not SOURCE_LABEL_DIR.exists():
    raise FileNotFoundError(f"Label folder not found: {SOURCE_LABEL_DIR}")

Training device: cpu
Validation device: cpu
Saved environment info: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/environment_info.json
Training device: cpu
Current notebook folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis
Image folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/train/images
Label folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/train/labels
Image folder exists: True
Label folder exists: True
Number of JPG images: 1308
Number of TXT labels: 1308


In [2]:
# Cell 1B — Optional cleanup before rerun

CLEAN_PREVIOUS_OUTPUTS = True   # Change to False if you want to keep previous outputs

def safe_delete_path(path_to_delete, dataset_root):
    """
    Safely delete only paths inside the project folder.
    This prevents accidental deletion of raw source data or external folders.
    """
    path_to_delete = Path(path_to_delete).resolve()
    dataset_root = Path(dataset_root).resolve()

    if not str(path_to_delete).startswith(str(dataset_root)):
        raise ValueError(f"Unsafe delete blocked: {path_to_delete}")

    if path_to_delete.exists():
        if path_to_delete.is_dir():
            shutil.rmtree(path_to_delete)
            print("Deleted folder:", path_to_delete)
        else:
            path_to_delete.unlink()
            print("Deleted file:", path_to_delete)


if CLEAN_PREVIOUS_OUTPUTS:
    # Delete recreated working split folder
    safe_delete_path(WORK_DIR, DATASET_ROOT)

    # Delete all previous generated outputs, including old YOLO runs
    safe_delete_path(OUTPUT_DIR, DATASET_ROOT)

    # Delete mapping files created at project root
    safe_delete_path(TREE_MAPPING_CSV, DATASET_ROOT)
    safe_delete_path(TREE_GPS_REFERENCE_CSV, DATASET_ROOT)

    # Recreate output folder after deletion
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Previous generated outputs cleaned successfully.")
else:
    print("Previous outputs retained.")

Deleted folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working
Deleted folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs
Deleted file: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/image_tree_mapping.csv
Deleted file: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/tree_gps_reference.csv
Previous generated outputs cleaned successfully.


In [3]:
# Cell 2 — Dynamically create drone-based tree mapping from actual image filenames

import math
from pathlib import Path

IMAGE_EXTENSIONS = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]


def meters_to_latitude_delta(meters):
    return meters / 111320


def meters_to_longitude_delta(meters, latitude):
    return meters / (111320 * math.cos(math.radians(latitude)))


def get_all_image_files(source_image_dir):
    image_files = []

    for ext in IMAGE_EXTENSIONS:
        image_files.extend(list(source_image_dir.glob(ext)))

    image_files = sorted(image_files, key=lambda x: x.name.lower())
    return image_files


def split_count_evenly(total_count, number_of_groups):
    """
    Example:
    total_count = 1308, number_of_groups = 5
    output = [262, 262, 262, 261, 261]
    """
    base = total_count // number_of_groups
    remainder = total_count % number_of_groups

    return [
        base + 1 if i < remainder else base
        for i in range(number_of_groups)
    ]


def create_dynamic_drone_tree_mapping():
    image_files = get_all_image_files(SOURCE_IMAGE_DIR)

    if len(image_files) == 0:
        raise ValueError(f"No images found in: {SOURCE_IMAGE_DIR}")

    total_images = len(image_files)

    print("Images found:", total_images)
    print("Image folder:", SOURCE_IMAGE_DIR)
    print("Label folder:", SOURCE_LABEL_DIR)

    orchard_counts = split_count_evenly(total_images, N_ORCHARDS)

    # Placeholder orchard center GPS coordinates.
    # Replace later with actual orchard center GPS if available.
    orchard_centers = {}

    base_latitude = 26.700000
    base_longitude = 88.400000

    for orchard_index in range(1, N_ORCHARDS + 1):
        orchard_id = f"ORCHARD_{orchard_index:02d}"

        orchard_centers[orchard_id] = {
            "latitude": base_latitude + ((orchard_index - 1) * 0.010000),
            "longitude": base_longitude + ((orchard_index - 1) * 0.010000)
        }

    rows = []

    global_tree_sequence = 1
    image_index = 0

    for orchard_index, trees_in_orchard in enumerate(orchard_counts, start=1):
        orchard_id = f"ORCHARD_{orchard_index:02d}"

        orchard_lat = orchard_centers[orchard_id]["latitude"]
        orchard_lon = orchard_centers[orchard_id]["longitude"]

        block_counts = split_count_evenly(trees_in_orchard, BLOCKS_PER_ORCHARD)

        tree_sequence_in_orchard = 1

        for block_index, trees_in_block in enumerate(block_counts, start=1):
            block_id = f"BLOCK_{block_index:02d}"

            row_counts = split_count_evenly(trees_in_block, ROWS_PER_BLOCK)

            tree_sequence_in_block = 1

            for row_index, trees_in_row in enumerate(row_counts, start=1):
                row_id = f"ROW_{row_index:02d}"

                for tree_sequence_in_row in range(1, trees_in_row + 1):
                    if image_index >= total_images:
                        break

                    img_path = image_files[image_index]
                    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"

                    has_label = label_path.exists()

                    tree_id = (
                        f"O{orchard_index:02d}"
                        f"_B{block_index:02d}"
                        f"_R{row_index:02d}"
                        f"_T{tree_sequence_in_row:03d}"
                    )

                    tree_global_id = tree_id

                    # Simulated orchard layout in meters.
                    # This is drone-map friendly because it creates local X/Y positions.
                    local_x_m = (
                        (block_index - 1) * 120
                        + (tree_sequence_in_row - 1) * 5
                    )

                    local_y_m = (
                        (row_index - 1) * 8
                        + (block_index - 1) * 40
                    )

                    tree_latitude = orchard_lat + meters_to_latitude_delta(local_y_m)
                    tree_longitude = orchard_lon + meters_to_longitude_delta(local_x_m, orchard_lat)

                    # Drone camera position is slightly above/near the ground-projected tree point.
                    # In real drone mapping, these should come from drone image EXIF / flight log.
                    drone_altitude_m = 35.0
                    relative_altitude_m = 35.0

                    ground_projection_latitude = tree_latitude
                    ground_projection_longitude = tree_longitude

                    drone_camera_latitude = tree_latitude + meters_to_latitude_delta(1.5)
                    drone_camera_longitude = tree_longitude + meters_to_longitude_delta(1.5, tree_latitude)

                    image_latitude = drone_camera_latitude
                    image_longitude = drone_camera_longitude

                    # Since this is simulated mapping, distance is small and fixed.
                    gps_match_distance_m = 2.1

                    if gps_match_distance_m <= 5:
                        gps_match_status = "DRONE_MATCHED"
                    elif gps_match_distance_m <= 10:
                        gps_match_status = "DRONE_LOW_CONFIDENCE"
                    else:
                        gps_match_status = "DRONE_REVIEW_REQUIRED"

                    # Simulated orthomosaic pixel positions.
                    # Later this can come from actual drone orthomosaic processing.
                    orthomosaic_pixel_x = int(local_x_m / 0.05)
                    orthomosaic_pixel_y = int(local_y_m / 0.05)

                    rows.append({
                        "image_name": img_path.name,
                        "label_name": label_path.name,
                        "image_path": str(img_path),
                        "label_path": str(label_path),
                        "has_label": has_label,

                        "mission_id": "MISSION_001",
                        "flight_id": f"FLIGHT_ORCHARD_{orchard_index:02d}",
                        "drone_id": "DRONE_001",
                        "orthomosaic_id": f"ORTHO_ORCHARD_{orchard_index:02d}",

                        "orchard_id": orchard_id,
                        "block_id": block_id,
                        "row_id": row_id,
                        "tree_id": tree_id,
                        "tree_global_id": tree_global_id,
                        "tree_group_id": tree_global_id,

                        "tree_sequence_global": global_tree_sequence,
                        "tree_sequence_in_orchard": tree_sequence_in_orchard,
                        "tree_sequence_in_block": tree_sequence_in_block,
                        "tree_sequence_in_row": tree_sequence_in_row,

                        "tree_latitude": tree_latitude,
                        "tree_longitude": tree_longitude,
                        "tree_centroid_latitude": tree_latitude,
                        "tree_centroid_longitude": tree_longitude,
                        "tree_canopy_radius_m": 2.5,

                        "image_latitude": image_latitude,
                        "image_longitude": image_longitude,
                        "drone_camera_latitude": drone_camera_latitude,
                        "drone_camera_longitude": drone_camera_longitude,
                        "drone_altitude_m": drone_altitude_m,
                        "relative_altitude_m": relative_altitude_m,

                        "camera_heading_deg": 0,
                        "drone_yaw_deg": 0,
                        "gimbal_pitch_deg": -90,
                        "gsd_cm_per_px": 5.0,

                        "ground_projection_latitude": ground_projection_latitude,
                        "ground_projection_longitude": ground_projection_longitude,
                        "orthomosaic_pixel_x": orthomosaic_pixel_x,
                        "orthomosaic_pixel_y": orthomosaic_pixel_y,
                        "local_x_m": local_x_m,
                        "local_y_m": local_y_m,

                        "gps_match_distance_m": gps_match_distance_m,
                        "gps_match_status": gps_match_status,
                        "gps_accuracy_m": 3.0,
                        "coordinate_system": "WGS84",
                        "gps_source": "SIMULATED_DRONE_MAPPING",
                        "mapping_method": "DYNAMIC_FROM_IMAGE_FOLDER",
                        "is_simulated_gps": True,

                        "capture_date": "UNKNOWN_DATE",
                        "view_angle": "drone_nadir"
                    })

                    global_tree_sequence += 1
                    tree_sequence_in_orchard += 1
                    tree_sequence_in_block += 1
                    image_index += 1

    image_tree_mapping_df = pd.DataFrame(rows)

    tree_gps_reference_df = image_tree_mapping_df[
        [
            "orchard_id",
            "block_id",
            "row_id",
            "tree_id",
            "tree_global_id",
            "tree_sequence_global",
            "tree_sequence_in_orchard",
            "tree_sequence_in_block",
            "tree_sequence_in_row",
            "tree_latitude",
            "tree_longitude",
            "tree_centroid_latitude",
            "tree_centroid_longitude",
            "tree_canopy_radius_m",
            "orthomosaic_id",
            "orthomosaic_pixel_x",
            "orthomosaic_pixel_y",
            "local_x_m",
            "local_y_m",
            "coordinate_system",
            "gps_source",
            "mapping_method",
            "is_simulated_gps"
        ]
    ].copy()

    image_tree_mapping_df.to_csv(TREE_MAPPING_CSV, index=False)
    tree_gps_reference_df.to_csv(TREE_GPS_REFERENCE_CSV, index=False)

    print("Created:", TREE_MAPPING_CSV)
    print("Created:", TREE_GPS_REFERENCE_CSV)
    print("Total mapped images:", len(image_tree_mapping_df))
    print("Images without matching label txt:", image_tree_mapping_df["has_label"].value_counts().to_dict())

    return image_tree_mapping_df, tree_gps_reference_df


if RECREATE_TREE_MAPPING_EACH_RUN or not TREE_MAPPING_CSV.exists() or not TREE_GPS_REFERENCE_CSV.exists():
    tree_mapping_df, tree_gps_reference_df = create_dynamic_drone_tree_mapping()
else:
    tree_mapping_df = pd.read_csv(TREE_MAPPING_CSV)
    tree_gps_reference_df = pd.read_csv(TREE_GPS_REFERENCE_CSV)
    print("Loaded existing mapping files.")


TREE_METADATA_COLUMNS = [
    "mission_id",
    "flight_id",
    "drone_id",
    "orthomosaic_id",

    "orchard_id",
    "block_id",
    "row_id",
    "tree_id",
    "tree_global_id",
    "tree_group_id",

    "tree_sequence_global",
    "tree_sequence_in_orchard",
    "tree_sequence_in_block",
    "tree_sequence_in_row",

    "tree_latitude",
    "tree_longitude",
    "tree_centroid_latitude",
    "tree_centroid_longitude",
    "tree_canopy_radius_m",

    "image_latitude",
    "image_longitude",
    "drone_camera_latitude",
    "drone_camera_longitude",
    "drone_altitude_m",
    "relative_altitude_m",

    "camera_heading_deg",
    "drone_yaw_deg",
    "gimbal_pitch_deg",
    "gsd_cm_per_px",

    "ground_projection_latitude",
    "ground_projection_longitude",
    "orthomosaic_pixel_x",
    "orthomosaic_pixel_y",
    "local_x_m",
    "local_y_m",

    "gps_match_distance_m",
    "gps_match_status",
    "gps_accuracy_m",
    "coordinate_system",
    "gps_source",
    "mapping_method",
    "is_simulated_gps",

    "capture_date",
    "view_angle"
]


tree_mapping_df["image_name"] = tree_mapping_df["image_name"].astype(str)

tree_metadata_lookup = (
    tree_mapping_df
    .set_index("image_name")[TREE_METADATA_COLUMNS]
    .to_dict(orient="index")
)


def get_tree_metadata(image_name):
    default_metadata = {
        col: np.nan for col in TREE_METADATA_COLUMNS
    }

    default_metadata.update({
        "orchard_id": "UNKNOWN_ORCHARD",
        "block_id": "UNKNOWN_BLOCK",
        "row_id": "UNKNOWN_ROW",
        "tree_id": "UNKNOWN_TREE",
        "tree_global_id": "UNKNOWN_TREE",
        "tree_group_id": "UNKNOWN_TREE",
        "gps_match_status": "NO_MAPPING",
        "gps_source": "NO_MAPPING",
        "mapping_method": "NO_MAPPING",
        "is_simulated_gps": True,
        "capture_date": "UNKNOWN_DATE",
        "view_angle": "UNKNOWN_VIEW"
    })

    return tree_metadata_lookup.get(image_name, default_metadata)


display(tree_mapping_df.head())

Images found: 1308
Image folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/train/images
Label folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/train/labels
Created: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/image_tree_mapping.csv
Created: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/tree_gps_reference.csv
Total mapped images: 1308
Images without matching label txt: {True: 1308}


,image_name,label_name,image_path,label_path,has_label,mission_id,flight_id,drone_id,orthomosaic_id,orchard_id,...,local_y_m,gps_match_distance_m,gps_match_status,gps_accuracy_m,coordinate_system,gps_source,mapping_method,is_simulated_gps,capture_date,view_angle
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.txt,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,True,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,...,0,2.1,DRONE_MATCHED,3.0,WGS84,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True,UNKNOWN_DATE,drone_nadir
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.txt,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,True,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,...,0,2.1,DRONE_MATCHED,3.0,WGS84,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True,UNKNOWN_DATE,drone_nadir
2,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.txt,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,True,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,...,0,2.1,DRONE_MATCHED,3.0,WGS84,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True,UNKNOWN_DATE,drone_nadir
3,-102-_jpg.rf.5af9c7ede77bee134dcb5811dd2efe44.jpg,-102-_jpg.rf.5af9c7ede77bee134dcb5811dd2efe44.txt,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,True,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,...,0,2.1,DRONE_MATCHED,3.0,WGS84,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True,UNKNOWN_DATE,drone_nadir
4,-104-_jpg.rf.243a3ea518ad9685625bfe5d9dd0ea40.jpg,-104-_jpg.rf.243a3ea518ad9685625bfe5d9dd0ea40.txt,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,True,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,...,0,2.1,DRONE_MATCHED,3.0,WGS84,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True,UNKNOWN_DATE,drone_nadir


In [4]:
# Cell 3 — Read YOLO labels and audit dataset
def read_yolo_label(label_path):
    """
    Reads a YOLO txt label file.
    Returns rows: class_id, x_center, y_center, width, height
    """
    rows = []
    if not label_path.exists():
        return rows
    
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = parts
            rows.append({
                "class_id": int(float(cls)),
                "x_center": float(xc),
                "y_center": float(yc),
                "bbox_width": float(w),
                "bbox_height": float(h)
            })
    return rows


image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
image_paths = []
for ext in image_extensions:
    image_paths.extend(list(SOURCE_IMAGE_DIR.glob(ext)))

# Keep image ordering stable and aligned with dynamic mapping logic
image_paths = sorted(image_paths, key=lambda x: x.name.lower())

records = []

for img_path in tqdm(image_paths, desc="Auditing labels"):
    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
    labels = read_yolo_label(label_path)
    tree_meta = get_tree_metadata(img_path.name)
    
    img = cv2.imread(str(img_path))
    if img is None:
        img_h, img_w = None, None
    else:
        img_h, img_w = img.shape[:2]
    
    if len(labels) == 0:
        records.append({
            "image_path": str(img_path),
            "image_name": img_path.name,
            "label_path": str(label_path),
            **tree_meta,
            "has_label": False,
            "objects_in_image": 0,
            "class_id": None,
            "class_name": None,
            "x_center": None,
            "y_center": None,
            "bbox_width": None,
            "bbox_height": None,
            "image_width": img_w,
            "image_height": img_h
        })
    else:
        for lab in labels:
            cls_id = lab["class_id"]
            records.append({
                "image_path": str(img_path),
                "image_name": img_path.name,
                "label_path": str(label_path),
                **tree_meta,
                "has_label": True,
                "objects_in_image": len(labels),
                "class_id": cls_id,
                "class_name": CLASS_NAMES.get(cls_id, "UNKNOWN"),
                "x_center": lab["x_center"],
                "y_center": lab["y_center"],
                "bbox_width": lab["bbox_width"],
                "bbox_height": lab["bbox_height"],
                "image_width": img_w,
                "image_height": img_h
            })

label_audit_df = pd.DataFrame(records)
label_audit_path = OUTPUT_DIR / "dataset_label_audit.csv"
label_audit_df.to_csv(label_audit_path, index=False)

print("Total images:", len(image_paths))
print("Total labelled objects:", label_audit_df["has_label"].sum())
print("Saved:", label_audit_path)

display(label_audit_df.head())

Auditing labels: 100%|██████████| 1308/1308 [00:02<00:00, 567.55it/s]


Total images: 1308
Total labelled objects: 15759
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/dataset_label_audit.csv


,image_path,image_name,label_path,mission_id,flight_id,drone_id,orthomosaic_id,orchard_id,block_id,row_id,...,has_label,objects_in_image,class_id,class_name,x_center,y_center,bbox_width,bbox_height,image_width,image_height
0,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,True,1,0.0,Green,0.577344,0.620313,0.148438,0.143750,640,640
1,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,True,10,1.0,Half,0.541406,0.402344,0.248438,0.260937,640,640
2,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,True,10,1.0,Half,0.760938,0.502344,0.231250,0.282813,640,640
3,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,True,10,1.0,Half,0.351562,0.389844,0.206250,0.248438,640,640
4,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,True,10,1.0,Half,0.483594,0.578125,0.195312,0.187500,640,640


In [5]:
# Cell 4 — Dataset statistics before training
dataset_stats = []

total_images = len(image_paths)
total_label_files = len(list(SOURCE_LABEL_DIR.glob("*.txt")))
total_objects = label_audit_df[label_audit_df["has_label"] == True].shape[0]
images_without_labels = label_audit_df[label_audit_df["has_label"] == False]["image_name"].nunique()

dataset_stats.append({
    "metric": "total_images",
    "value": total_images
})
dataset_stats.append({
    "metric": "total_label_files",
    "value": total_label_files
})
dataset_stats.append({
    "metric": "total_objects",
    "value": total_objects
})
dataset_stats.append({
    "metric": "images_without_labels",
    "value": images_without_labels
})
dataset_stats.append({
    "metric": "avg_objects_per_image",
    "value": total_objects / max(total_images, 1)
})

class_distribution = (
    label_audit_df[label_audit_df["has_label"] == True]
    .groupby(["class_id", "class_name"])
    .size()
    .reset_index(name="object_count")
)

for _, row in class_distribution.iterrows():
    dataset_stats.append({
        "metric": f"class_{int(row['class_id'])}_{row['class_name']}_object_count",
        "value": row["object_count"]
    })

dataset_stats_df = pd.DataFrame(dataset_stats)
dataset_stats_path = OUTPUT_DIR / "dataset_statistics.csv"
class_distribution_path = OUTPUT_DIR / "class_distribution.csv"

dataset_stats_df.to_csv(dataset_stats_path, index=False)
class_distribution.to_csv(class_distribution_path, index=False)

print("Saved:", dataset_stats_path)
print("Saved:", class_distribution_path)

display(dataset_stats_df)
display(class_distribution)

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/dataset_statistics.csv
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/class_distribution.csv


,metric,value
0,total_images,1308.000000
1,total_label_files,1308.000000
2,total_objects,15759.000000
3,images_without_labels,1.000000
4,avg_objects_per_image,12.048165
5,class_0_Green_object_count,4210.000000
6,class_1_Half_object_count,1029.000000
7,class_2_Red_object_count,6989.000000
8,class_3_Young_object_count,3531.000000


,class_id,class_name,object_count
0,0.0,Green,4210
1,1.0,Half,1029
2,2.0,Red,6989
3,3.0,Young,3531


In [6]:
# Cell 4A — Clean duplicate YOLO labels before train/validation split

def remove_duplicate_yolo_labels(label_dir):
    label_dir = Path(label_dir)
    cleaned_files = 0
    duplicate_lines_removed = 0

    for txt_file in label_dir.glob("*.txt"):
        with open(txt_file, "r") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

        unique_lines = list(dict.fromkeys(lines))

        if len(unique_lines) < len(lines):
            duplicate_lines_removed += len(lines) - len(unique_lines)
            cleaned_files += 1

            with open(txt_file, "w") as f:
                for line in unique_lines:
                    f.write(line + "\n")

    print("Label files cleaned:", cleaned_files)
    print("Duplicate label lines removed:", duplicate_lines_removed)

remove_duplicate_yolo_labels(SOURCE_LABEL_DIR)

Label files cleaned: 0
Duplicate label lines removed: 0


In [7]:
# Cell 5 — Create train/validation split

def get_primary_class_for_image(img_path):
    """
    Returns the dominant class in an image.
    If the image has multiple fruit objects, this uses the most frequent class.
    """
    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
    labels = read_yolo_label(label_path)

    if len(labels) == 0:
        return -1

    class_ids = [lab["class_id"] for lab in labels]
    values, counts = np.unique(class_ids, return_counts=True)

    dominant_class = values[np.argmax(counts)]
    return int(dominant_class)
    

# Use only images that have matching labels
valid_images = []
primary_classes = []

for img_path in image_paths:
    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
    if label_path.exists() and len(read_yolo_label(label_path)) > 0:
        valid_images.append(img_path)
        primary_classes.append(get_primary_class_for_image(img_path))

valid_images = np.array(valid_images)
primary_classes = np.array(primary_classes)

# Stratified split if possible
unique_classes, counts = np.unique(primary_classes, return_counts=True)
can_stratify = np.all(counts >= 2) and len(unique_classes) > 1

train_imgs, val_imgs = train_test_split(
    valid_images,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=primary_classes if can_stratify else None
)

print("Train images:", len(train_imgs))
print("Validation images:", len(val_imgs))
print("Stratified split used:", can_stratify)

# Clear old YOLO working directory before recreating train/val split
# This prevents old files from previous runs from mixing with the new split.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

print("Old working directory cleared:", WORK_DIR)

# Create folders
for split in ["train", "val"]:
    (WORK_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (WORK_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def copy_image_and_label(img_path, split):
    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
    
    dst_img = WORK_DIR / "images" / split / img_path.name
    dst_label = WORK_DIR / "labels" / split / label_path.name
    
    shutil.copy2(img_path, dst_img)
    shutil.copy2(label_path, dst_label)


for img_path in tqdm(train_imgs, desc="Copying train"):
    copy_image_and_label(img_path, "train")

for img_path in tqdm(val_imgs, desc="Copying val"):
    copy_image_and_label(img_path, "val")

print("YOLO working directory created:", WORK_DIR)

# Check train/validation object-level class distribution
def count_classes_in_split(image_list):
    counts = {class_id: 0 for class_id in CLASS_NAMES.keys()}

    for img_path in image_list:
        label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
        labels = read_yolo_label(label_path)

        for lab in labels:
            counts[lab["class_id"]] += 1

    return {
        CLASS_NAMES[class_id]: count
        for class_id, count in counts.items()
    }

train_class_counts = count_classes_in_split(train_imgs)
val_class_counts = count_classes_in_split(val_imgs)

print("Train class counts:", train_class_counts)
print("Validation class counts:", val_class_counts)

Train images: 1045
Validation images: 262
Stratified split used: True
Old working directory cleared: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working


Copying val: 100%|██████████| 262/262 [00:00<00:00, 1106.90it/s]


YOLO working directory created: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working
Train class counts: {'Green': 3305, 'Half': 842, 'Red': 5620, 'Young': 2690}
Validation class counts: {'Green': 905, 'Half': 187, 'Red': 1369, 'Young': 841}


In [8]:
# Cell 6 — Create YOLO dataset YAML
data_yaml = {
    "path": str(WORK_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "Green",
        1: "Half",
        2: "Red",
        3: "Young"
    }
}

yaml_path = WORK_DIR / "lichi_maturity.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Created YAML:", yaml_path)

with open(yaml_path, "r") as f:
    print(f.read())

Created YAML: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working/lichi_maturity.yaml
path: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image
  _Analysis/yolo_research_working
train: images/train
val: images/val
names:
  0: Green
  1: Half
  2: Red
  3: Young



In [9]:
# Cell 7 — Train YOLO maturity detection model
model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    project=str(OUTPUT_DIR / "yolo_runs"),
    name="lichi_maturity_detector",
    patience=25,
    plots=True,
    max_det=YOLO_MAX_DET,
    iou=YOLO_NMS_IOU,
    workers=0,

    # Important: disable validation during training on MPS
    val=False
)

best_model_path = Path(train_results.save_dir) / "weights" / "best.pt"
last_model_path = Path(train_results.save_dir) / "weights" / "last.pt"

if best_model_path.exists():
    model_to_use_path = best_model_path
else:
    model_to_use_path = last_model_path

print("Best model:", best_model_path)
print("Last model:", last_model_path)
print("Model selected for validation/prediction:", model_to_use_path)


Ultralytics 8.4.48 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working/lichi_maturity.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=100, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale

In [10]:
# ==========================================================
# Cell 8 - Save YOLO epoch-wise training metrics
# Note: validation metrics are generated separately in Cell 9 because val=False in Cell 7.
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ----------------------------------------------------------
# 1. Locate YOLO results.csv
# ----------------------------------------------------------
# train_results.save_dir is created by YOLO during training.
# It usually contains:
# weights/
# results.csv
# confusion_matrix.png
# results.png
# PR_curve.png, etc.

run_dir = Path(train_results.save_dir)
yolo_results_csv = run_dir / "results.csv"

if not yolo_results_csv.exists():
    raise FileNotFoundError(f"YOLO results.csv not found at: {yolo_results_csv}")

print("YOLO results file found:", yolo_results_csv)

# ----------------------------------------------------------
# 2. Read YOLO results.csv
# ----------------------------------------------------------

raw_training_metrics_df = pd.read_csv(yolo_results_csv)

# Clean column names because YOLO sometimes adds spaces
raw_training_metrics_df.columns = [
    c.strip() for c in raw_training_metrics_df.columns
]

print("Available YOLO metric columns:")
print(raw_training_metrics_df.columns.tolist())

# ----------------------------------------------------------
# 3. Create research-friendly epoch metrics dataframe
# ----------------------------------------------------------

def get_col(df, possible_names):
    """
    Returns the first matching column from possible_names.
    If no column is found, returns NaN.
    """
    for name in possible_names:
        if name in df.columns:
            return df[name]
    return np.nan


epoch_metrics_df = pd.DataFrame()

epoch_metrics_df["epoch"] = get_col(
    raw_training_metrics_df,
    ["epoch"]
)

# Training losses
epoch_metrics_df["train_box_loss"] = get_col(
    raw_training_metrics_df,
    ["train/box_loss", "box_loss"]
)

epoch_metrics_df["train_cls_loss"] = get_col(
    raw_training_metrics_df,
    ["train/cls_loss", "cls_loss"]
)

epoch_metrics_df["train_dfl_loss"] = get_col(
    raw_training_metrics_df,
    ["train/dfl_loss", "dfl_loss"]
)

# Validation metrics
epoch_metrics_df["precision_box"] = get_col(
    raw_training_metrics_df,
    ["metrics/precision(B)", "precision"]
)

epoch_metrics_df["recall_box"] = get_col(
    raw_training_metrics_df,
    ["metrics/recall(B)", "recall"]
)

epoch_metrics_df["mAP50_box"] = get_col(
    raw_training_metrics_df,
    ["metrics/mAP50(B)", "mAP50"]
)

epoch_metrics_df["mAP50_95_box"] = get_col(
    raw_training_metrics_df,
    ["metrics/mAP50-95(B)", "mAP50-95"]
)

# Validation losses, if available
epoch_metrics_df["val_box_loss"] = get_col(
    raw_training_metrics_df,
    ["val/box_loss"]
)

epoch_metrics_df["val_cls_loss"] = get_col(
    raw_training_metrics_df,
    ["val/cls_loss"]
)

epoch_metrics_df["val_dfl_loss"] = get_col(
    raw_training_metrics_df,
    ["val/dfl_loss"]
)

# Learning rates, useful for research reproducibility
epoch_metrics_df["learning_rate_pg0"] = get_col(
    raw_training_metrics_df,
    ["lr/pg0"]
)

epoch_metrics_df["learning_rate_pg1"] = get_col(
    raw_training_metrics_df,
    ["lr/pg1"]
)

epoch_metrics_df["learning_rate_pg2"] = get_col(
    raw_training_metrics_df,
    ["lr/pg2"]
)

# ----------------------------------------------------------
# 4. Add static training context columns
# ----------------------------------------------------------

epoch_metrics_df["image_size"] = IMAGE_SIZE
epoch_metrics_df["batch_size"] = BATCH_SIZE
epoch_metrics_df["base_model"] = BASE_MODEL
epoch_metrics_df["device"] = DEVICE

# Count images and label instances from working train/val folders
def count_images(folder):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    total = 0
    for ext in exts:
        total += len(list(Path(folder).glob(ext)))
    return total

def count_yolo_instances(label_folder):
    total_instances = 0
    label_folder = Path(label_folder)

    for txt_file in label_folder.glob("*.txt"):
        with open(txt_file, "r") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
            total_instances += len(lines)

    return total_instances

train_image_count = count_images(WORK_DIR / "images" / "train")
val_image_count = count_images(WORK_DIR / "images" / "val")

train_instance_count = count_yolo_instances(WORK_DIR / "labels" / "train")
val_instance_count = count_yolo_instances(WORK_DIR / "labels" / "val")

epoch_metrics_df["train_images"] = train_image_count
epoch_metrics_df["val_images"] = val_image_count
epoch_metrics_df["train_instances"] = train_instance_count
epoch_metrics_df["val_instances"] = val_instance_count

# ----------------------------------------------------------
# 5. Save clean metrics CSV
# ----------------------------------------------------------

epoch_metrics_path = OUTPUT_DIR / "epoch_training_validation_metrics.csv"
epoch_metrics_df.to_csv(epoch_metrics_path, index=False)

print("Saved:", epoch_metrics_path)
display(epoch_metrics_df.tail())

YOLO results file found: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/yolo_runs/lichi_maturity_detector/results.csv
Available YOLO metric columns:
['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/epoch_training_validation_metrics.csv


,epoch,train_box_loss,train_cls_loss,train_dfl_loss,precision_box,recall_box,mAP50_box,mAP50_95_box,val_box_loss,val_cls_loss,...,learning_rate_pg1,learning_rate_pg2,image_size,batch_size,base_model,device,train_images,val_images,train_instances,val_instances
45,46,0.93567,0.58995,1.00225,0.0000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.000136,0.000136,640,4,yolo11n.pt,cpu,1045,262,12457,3302
46,47,0.92112,0.57278,0.99265,0.0000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.000111,0.000111,640,4,yolo11n.pt,cpu,1045,262,12457,3302
47,48,0.91659,0.57589,0.99024,0.0000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.000087,0.000087,640,4,yolo11n.pt,cpu,1045,262,12457,3302
48,49,0.90611,0.55967,0.99036,0.0000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.000062,0.000062,640,4,yolo11n.pt,cpu,1045,262,12457,3302
49,50,0.91081,0.56540,0.98770,0.8523,0.81739,0.89763,0.65529,0.89736,0.64604,...,0.000037,0.000037,640,4,yolo11n.pt,cpu,1045,262,12457,3302


In [11]:
# Cell 9 — Validate model and save research metrics


trained_model = YOLO(str(model_to_use_path))

metrics = trained_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=VAL_DEVICE,
    plots=True,

    # Fix for slow NMS during validation
    max_det=YOLO_MAX_DET,
    iou=YOLO_NMS_IOU
)

summary_metrics = {
    "precision_mean": float(getattr(metrics.box, "mp", np.nan)),
    "recall_mean": float(getattr(metrics.box, "mr", np.nan)),
    "mAP50": float(getattr(metrics.box, "map50", np.nan)),
    "mAP50_95": float(getattr(metrics.box, "map", np.nan)),
    "mAP75": float(getattr(metrics.box, "map75", np.nan)),
    "fitness": float(getattr(metrics, "fitness", np.nan)) if not callable(getattr(metrics, "fitness", None)) else np.nan
}

metrics_summary_df = pd.DataFrame([summary_metrics])
metrics_summary_path = OUTPUT_DIR / "validation_metrics_summary.csv"
metrics_summary_df.to_csv(metrics_summary_path, index=False)

print("Saved:", metrics_summary_path)
display(metrics_summary_df)


# Per-class mAP50-95 if available
per_class_rows = []
class_maps = getattr(metrics.box, "maps", None)

if class_maps is not None:
    for cls_id, map_value in enumerate(class_maps):
        per_class_rows.append({
            "class_id": cls_id,
            "class_name": CLASS_NAMES.get(cls_id, "UNKNOWN"),
            "mAP50_95": float(map_value)
        })

per_class_metrics_df = pd.DataFrame(per_class_rows)
per_class_metrics_path = OUTPUT_DIR / "per_class_validation_metrics.csv"
per_class_metrics_df.to_csv(per_class_metrics_path, index=False)

print("Saved:", per_class_metrics_path)
display(per_class_metrics_df)

Ultralytics 8.4.48 🚀 Python-3.9.6 torch-2.8.0 CPU (Apple M1)
YOLO11n summary (fused): 101 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 499.2±589.0 MB/s, size: 56.1 KB)
val: Scanning /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/yolo_research_working/labels/val.cache... 262 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 262/262 84.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 4.9it/s 13.4s0.2s
                   all        262       3302      0.853      0.817      0.898      0.655
                 Green        101        905      0.887      0.884      0.936        0.7
                  Half         46        187      0.765       0.69       0.79      0.676
                   Red         82       1369      0.915      0.803      0.917      0.618
                 Young   

,precision_mean,recall_mean,mAP50,mAP50_95,mAP75,fitness
0,0.85304,0.817394,0.897712,0.655085,0.750135,0.655085


Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/per_class_validation_metrics.csv


,class_id,class_name,mAP50_95
0,0,Green,0.700191
1,1,Half,0.676233
2,2,Red,0.617787
3,3,Young,0.626131


In [12]:
#  Cell 10 — Utility functions for additional image-level and fruit-level attributes
def yolo_xywh_to_xyxy(xc, yc, w, h, img_w, img_h):
    x1 = int((xc - w / 2) * img_w)
    y1 = int((yc - h / 2) * img_h)
    x2 = int((xc + w / 2) * img_w)
    y2 = int((yc + h / 2) * img_h)
    
    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))
    
    return x1, y1, x2, y2


def calculate_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    
    union = area_a + area_b - inter_area
    
    if union == 0:
        return 0.0
    
    return inter_area / union


def image_quality_metrics(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness_mean = gray.mean()
    contrast_std = gray.std()
    
    flags = []
    
    if blur_score < 80:
        flags.append("BLURRY")
    if brightness_mean < 45:
        flags.append("LOW_LIGHT")
    if brightness_mean > 215:
        flags.append("OVEREXPOSED")
    if contrast_std < 25:
        flags.append("LOW_CONTRAST")
    
    if len(flags) == 0:
        quality_flag = "OK"
    else:
        quality_flag = "|".join(flags)
    
    return {
        "blur_score": float(blur_score),
        "brightness_mean": float(brightness_mean),
        "contrast_std": float(contrast_std),
        "image_quality_flag": quality_flag
    }


def canopy_gap_pct_proxy(image_bgr):
    """
    Proxy: estimates bright non-foliage/open areas.
    Works best when sky/background gaps are visible.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    # Green foliage mask
    green_mask = ((h >= 35) & (h <= 90) & (s > 40) & (v > 40))
    
    # Bright low-saturation regions can indicate sky/open gap/high background
    bright_open_mask = ((v > 170) & (s < 80))
    
    total_pixels = image_bgr.shape[0] * image_bgr.shape[1]
    gap_pct = bright_open_mask.sum() / max(total_pixels, 1) * 100
    
    green_pct = green_mask.sum() / max(total_pixels, 1) * 100
    
    return float(gap_pct), float(green_pct)


def visible_stress_proxy(image_bgr):
    """
    Proxy for visible plant stress using yellow/brown/dry-looking pixels.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    yellow_mask = ((h >= 18) & (h <= 38) & (s > 50) & (v > 70))
    brown_mask = ((h >= 5) & (h <= 25) & (s > 40) & (v > 35) & (v < 170))
    dark_dry_mask = ((v < 45) & (s > 30))
    
    stress_mask = yellow_mask | brown_mask | dark_dry_mask
    
    stress_pct = stress_mask.sum() / max(image_bgr.shape[0] * image_bgr.shape[1], 1) * 100
    
    if stress_pct < 2:
        score = 0
    elif stress_pct < 5:
        score = 1
    elif stress_pct < 10:
        score = 2
    elif stress_pct < 18:
        score = 3
    elif stress_pct < 30:
        score = 4
    else:
        score = 5
    
    return float(stress_pct), int(score)


def flowering_score_proxy(image_bgr):
    """
    Weak proxy: detects small bright white/yellow areas.
    This can confuse flowers with highlights.
    Best replaced later with labelled flower detector.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    white_flower_like = ((v > 180) & (s < 70))
    yellow_flower_like = ((h >= 18) & (h <= 38) & (s > 60) & (v > 120))
    
    flower_like_mask = white_flower_like | yellow_flower_like
    
    flower_pct = flower_like_mask.sum() / max(image_bgr.shape[0] * image_bgr.shape[1], 1) * 100
    
    if flower_pct < 0.5:
        score = 0
    elif flower_pct < 1:
        score = 1
    elif flower_pct < 2:
        score = 2
    elif flower_pct < 4:
        score = 3
    elif flower_pct < 7:
        score = 4
    else:
        score = 5
    
    return float(flower_pct), int(score)


def discoloration_score_proxy(crop_bgr):
    """
    Fruit-crop proxy for discoloration/damage.
    Uses dark, brownish, and highly irregular color areas.
    """
    if crop_bgr is None or crop_bgr.size == 0:
        return np.nan, np.nan
    
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    dark_mask = v < 45
    brown_mask = ((h >= 5) & (h <= 25) & (s > 50) & (v > 40) & (v < 180))
    abnormal_mask = dark_mask | brown_mask
    
    discoloration_pct = abnormal_mask.sum() / max(crop_bgr.shape[0] * crop_bgr.shape[1], 1) * 100
    
    if discoloration_pct < 2:
        score = 0
    elif discoloration_pct < 5:
        score = 1
    elif discoloration_pct < 10:
        score = 2
    elif discoloration_pct < 18:
        score = 3
    elif discoloration_pct < 30:
        score = 4
    else:
        score = 5
    
    return float(discoloration_pct), int(score)


def fruit_exposure_score_proxy(confidence, bbox_area_pct, crop_bgr):
    """
    Proxy using model confidence, bounding-box area, and crop brightness.
    This is not true occlusion estimation, but useful as an operational signal.
    """
    if crop_bgr is None or crop_bgr.size == 0:
        return np.nan
    
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    crop_brightness = gray.mean()
    
    score = 1
    
    if confidence >= 0.40:
        score += 1
    if confidence >= 0.65:
        score += 1
    if bbox_area_pct >= 0.5:
        score += 1
    if 40 <= crop_brightness <= 210:
        score += 1
    
    return int(min(score, 5))


def thermal_proxy_score(visible_stress_score, brightness_mean, green_pct):
    """
    RGB-only visual proxy. Not a real thermal measurement.
    """
    score = 0
    
    if visible_stress_score >= 2:
        score += 1
    if visible_stress_score >= 4:
        score += 1
    if brightness_mean > 180:
        score += 1
    if green_pct < 35:
        score += 1
    
    return int(min(score, 5))


def maturity_numeric_index(class_name):
    """
    Maturity index for research aggregation.
    Adjust values if agronomist defines better scale.
    """
    mapping = {
        "Young": 10,
        "Green": 35,
        "Half": 65,
        "Red": 90
    }
    return mapping.get(class_name, np.nan)

In [13]:
# Cell 11 — Run predictions and extract additional information

# For business output / full dataset scoring, keep it as it is.
all_images_for_prediction = []
for ext in image_extensions:
    all_images_for_prediction.extend(list(SOURCE_IMAGE_DIR.glob(ext)))

# Keep prediction image ordering stable and aligned with mapping file
all_images_for_prediction = sorted(all_images_for_prediction, key=lambda x: x.name.lower())
print("Prediction images:", len(all_images_for_prediction))

# For clean validation-only prediction Purpose, Use this
# all_images_for_prediction = []
# validation_image_dir = WORK_DIR / "images" / "val"

# for ext in image_extensions:
#     all_images_for_prediction.extend(list(validation_image_dir.glob(ext)))

# print("Prediction images:", len(all_images_for_prediction))

fruit_rows = []
image_rows = []

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

for img_path in tqdm(all_images_for_prediction, desc="Predicting and extracting features"):
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        continue
 
    img_h, img_w = image_bgr.shape[:2]
    tree_meta = get_tree_metadata(img_path.name)
    
    quality = image_quality_metrics(image_bgr)
    gap_pct, green_pct = canopy_gap_pct_proxy(image_bgr)
    stress_pct, stress_score = visible_stress_proxy(image_bgr)
    flower_pct, flowering_score = flowering_score_proxy(image_bgr)
    thermal_score = thermal_proxy_score(
        visible_stress_score=stress_score,
        brightness_mean=quality["brightness_mean"],
        green_pct=green_pct
    )
    

    results = trained_model.predict(
        source=str(img_path),
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=VAL_DEVICE,
        max_det=YOLO_MAX_DET,
        verbose=False
    )
    detections = results[0].boxes
    
    image_detection_count = 0
    maturity_counts = {name: 0 for name in CLASS_NAMES.values()}
    confidence_values = []
    
    if detections is not None and len(detections) > 0:
        for det_idx, box in enumerate(detections):
            xyxy = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            cls_name = CLASS_NAMES.get(cls_id, "UNKNOWN")
            
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            x1 = max(0, min(x1, img_w - 1))
            y1 = max(0, min(y1, img_h - 1))
            x2 = max(0, min(x2, img_w - 1))
            y2 = max(0, min(y2, img_h - 1))
            
            crop = image_bgr[y1:y2, x1:x2]
            
            bbox_w = max(0, x2 - x1)
            bbox_h = max(0, y2 - y1)
            bbox_area_pct = (bbox_w * bbox_h) / max(img_w * img_h, 1) * 100
            
            discoloration_pct, discoloration_score = discoloration_score_proxy(crop)
            exposure_score = fruit_exposure_score_proxy(conf, bbox_area_pct, crop)
            
            if crop is not None and crop.size > 0:
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crop_hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
                
                fruit_rgb_mean_r = float(crop_rgb[:, :, 0].mean())
                fruit_rgb_mean_g = float(crop_rgb[:, :, 1].mean())
                fruit_rgb_mean_b = float(crop_rgb[:, :, 2].mean())
                fruit_hue_mean = float(crop_hsv[:, :, 0].mean())
                fruit_saturation_mean = float(crop_hsv[:, :, 1].mean())
                fruit_value_mean = float(crop_hsv[:, :, 2].mean())
            else:
                fruit_rgb_mean_r = np.nan
                fruit_rgb_mean_g = np.nan
                fruit_rgb_mean_b = np.nan
                fruit_hue_mean = np.nan
                fruit_saturation_mean = np.nan
                fruit_value_mean = np.nan
            
            maturity_counts[cls_name] = maturity_counts.get(cls_name, 0) + 1
            confidence_values.append(conf)
            image_detection_count += 1
            
            fruit_rows.append({
                "image_name": img_path.name,
                "image_path": str(img_path),
                **tree_meta,
                "detection_id": det_idx,
                "predicted_class_id": cls_id,
                "predicted_maturity_class": cls_name,
                "confidence": conf,
                "maturity_index_0_100": maturity_numeric_index(cls_name),
                "bbox_x1": x1,
                "bbox_y1": y1,
                "bbox_x2": x2,
                "bbox_y2": y2,
                "bbox_width_px": bbox_w,
                "bbox_height_px": bbox_h,
                "bbox_area_pct": bbox_area_pct,
                "fruit_exposure_score": exposure_score,
                "discoloration_pct": discoloration_pct,
                "discoloration_score": discoloration_score,
                "fruit_rgb_mean_r": fruit_rgb_mean_r,
                "fruit_rgb_mean_g": fruit_rgb_mean_g,
                "fruit_rgb_mean_b": fruit_rgb_mean_b,
                "fruit_hue_mean": fruit_hue_mean,
                "fruit_saturation_mean": fruit_saturation_mean,
                "fruit_value_mean": fruit_value_mean,
                "image_blur_score": quality["blur_score"],
                "image_brightness_mean": quality["brightness_mean"],
                "image_contrast_std": quality["contrast_std"],
                "image_quality_flag": quality["image_quality_flag"]
            })
    
    if len(confidence_values) > 0:
        avg_conf = float(np.mean(confidence_values))
        max_conf = float(np.max(confidence_values))
        avg_maturity_index = np.nanmean([
            maturity_numeric_index(row["predicted_maturity_class"])
            for row in fruit_rows
            if row["image_name"] == img_path.name
        ])
    else:
        avg_conf = np.nan
        max_conf = np.nan
        avg_maturity_index = np.nan
    
    image_rows.append({
        "image_name": img_path.name,
        "image_path": str(img_path),
        "detected_fruit_count": image_detection_count,
        **tree_meta,
        "young_count": maturity_counts.get("Young", 0),
        "green_count": maturity_counts.get("Green", 0),
        "half_count": maturity_counts.get("Half", 0),
        "red_count": maturity_counts.get("Red", 0),
        "avg_detection_confidence": avg_conf,
        "max_detection_confidence": max_conf,
        "avg_maturity_index_0_100": avg_maturity_index,
        "flowering_pct_proxy": flower_pct,
        "flowering_score": flowering_score,
        "visible_stress_pct_proxy": stress_pct,
        "visible_stress_score": stress_score,
        "canopy_gap_pct": gap_pct,
        "green_canopy_pct_proxy": green_pct,
        "thermal_proxy_score": thermal_score,
        "blur_score": quality["blur_score"],
        "brightness_mean": quality["brightness_mean"],
        "contrast_std": quality["contrast_std"],
        "image_quality_flag": quality["image_quality_flag"]
    })

fruit_predictions_df = pd.DataFrame(fruit_rows)
image_predictions_df = pd.DataFrame(image_rows)

fruit_predictions_path = OUTPUT_DIR / "fruit_level_predictions_and_features.csv"
image_predictions_path = OUTPUT_DIR / "image_level_predictions_and_agro_features.csv"

fruit_predictions_df.to_csv(fruit_predictions_path, index=False)
image_predictions_df.to_csv(image_predictions_path, index=False)

print("Saved:", fruit_predictions_path)
print("Saved:", image_predictions_path)

display(fruit_predictions_df.head())
display(image_predictions_df.head())

Prediction images: 1308


Predicting and extracting features: 100%|██████████| 1308/1308 [01:15<00:00, 17.26it/s]


Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/fruit_level_predictions_and_features.csv
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/image_level_predictions_and_agro_features.csv


,image_name,image_path,mission_id,flight_id,drone_id,orthomosaic_id,orchard_id,block_id,row_id,tree_id,...,fruit_rgb_mean_r,fruit_rgb_mean_g,fruit_rgb_mean_b,fruit_hue_mean,fruit_saturation_mean,fruit_value_mean,image_blur_score,image_brightness_mean,image_contrast_std,image_quality_flag
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,...,74.538230,75.056102,19.254251,32.467785,195.045113,77.089184,718.525913,97.113867,98.205189,OK
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,95.989979,81.246356,27.842583,27.574210,176.444352,96.413908,295.916802,101.816184,93.868193,OK
2,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,73.456807,50.822302,21.402889,19.702725,172.806745,73.587931,295.916802,101.816184,93.868193,OK
3,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,94.070957,83.064083,27.651818,26.502991,180.996039,94.453783,295.916802,101.816184,93.868193,OK
4,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,57.953209,45.254966,16.889801,22.929335,166.955882,58.299274,295.916802,101.816184,93.868193,OK


,image_name,image_path,detected_fruit_count,mission_id,flight_id,drone_id,orthomosaic_id,orchard_id,block_id,row_id,...,flowering_score,visible_stress_pct_proxy,visible_stress_score,canopy_gap_pct,green_canopy_pct_proxy,thermal_proxy_score,blur_score,brightness_mean,contrast_std,image_quality_flag
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,5,50.885254,5,26.984131,19.424805,3,718.525913,97.113867,98.205189,OK
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,15,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,5,56.736572,5,25.551758,8.890137,3,295.916802,101.816184,93.868193,OK
2,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,5,59.577881,5,26.724854,4.084961,3,440.361513,81.194885,104.689353,OK
3,-102-_jpg.rf.5af9c7ede77bee134dcb5811dd2efe44.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,5,51.264893,5,25.729004,0.396240,3,409.433212,86.509185,103.816102,OK
4,-104-_jpg.rf.243a3ea518ad9685625bfe5d9dd0ea40.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,...,5,48.309814,5,26.326416,22.648682,3,451.822312,104.171042,94.121329,OK


In [14]:
# Cell 12 — Tree-level drone mapping summary

tree_group_columns = [
    "orchard_id",
    "block_id",
    "row_id",
    "tree_id",
    "tree_global_id",
    "mission_id",
    "flight_id",
    "orthomosaic_id"
]

tree_geo_columns = [
    "tree_latitude",
    "tree_longitude",
    "tree_centroid_latitude",
    "tree_centroid_longitude",
    "tree_canopy_radius_m",
    "drone_altitude_m",
    "gimbal_pitch_deg",
    "gsd_cm_per_px",
    "ground_projection_latitude",
    "ground_projection_longitude",
    "orthomosaic_pixel_x",
    "orthomosaic_pixel_y",
    "local_x_m",
    "local_y_m",
    "gps_match_distance_m",
    "gps_match_status",
    "gps_source",
    "mapping_method",
    "is_simulated_gps"
]

tree_group_columns = [col for col in tree_group_columns if col in image_predictions_df.columns]
tree_geo_columns = [col for col in tree_geo_columns if col in image_predictions_df.columns]

tree_agg_dict = {
    "image_name": "nunique",
    "detected_fruit_count": "sum",
    "young_count": "sum",
    "green_count": "sum",
    "half_count": "sum",
    "red_count": "sum",
    "avg_detection_confidence": "mean",
    "max_detection_confidence": "max",
    "avg_maturity_index_0_100": "mean",
    "flowering_score": "mean",
    "visible_stress_score": "mean",
    "canopy_gap_pct": "mean",
    "green_canopy_pct_proxy": "mean",
    "thermal_proxy_score": "mean",
    "blur_score": "mean",
    "brightness_mean": "mean",
    "contrast_std": "mean"
}

for col in tree_geo_columns:
    tree_agg_dict[col] = "first"

tree_level_prediction_summary_df = (
    image_predictions_df
    .groupby(tree_group_columns, dropna=False)
    .agg(tree_agg_dict)
    .reset_index()
    .rename(columns={
        "image_name": "total_images",
        "detected_fruit_count": "total_detected_fruits"
    })
)

tree_level_prediction_summary_path = OUTPUT_DIR / "tree_level_prediction_summary.csv"
tree_level_prediction_summary_df.to_csv(tree_level_prediction_summary_path, index=False)

print("Saved:", tree_level_prediction_summary_path)
print("Tree-level rows:", len(tree_level_prediction_summary_df))

display(tree_level_prediction_summary_df.head())

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/tree_level_prediction_summary.csv
Tree-level rows: 1308


,orchard_id,block_id,row_id,tree_id,tree_global_id,mission_id,flight_id,orthomosaic_id,total_images,total_detected_fruits,...,ground_projection_longitude,orthomosaic_pixel_x,orthomosaic_pixel_y,local_x_m,local_y_m,gps_match_distance_m,gps_match_status,gps_source,mapping_method,is_simulated_gps
0,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,O01_B01_R01_T001,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,1,...,88.400000,0,0,0,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
1,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,O01_B01_R01_T002,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,15,...,88.400050,100,0,5,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
2,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T003,O01_B01_R01_T003,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,1,...,88.400101,200,0,10,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
3,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T004,O01_B01_R01_T004,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,1,...,88.400151,300,0,15,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
4,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T005,O01_B01_R01_T005,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,1,...,88.400201,400,0,20,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True


In [15]:
# ==========================================================
# Cell 13 - Fruit-Level Visible Damage / Disease-Risk Proxy Estimation
# ==========================================================
# This is RGB-image-based symptom detection.
# It does not diagnose exact disease names.
# It estimates visible disease/damage risk from fruit crop regions.
# ==========================================================

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

def calculate_disease_pattern_features(crop_bgr):
    """
    Estimate visible disease/damage patterns from a fruit crop.
    Returns proxy metrics and risk score.
    """

    if crop_bgr is None or crop_bgr.size == 0:
        return {
            "dark_spot_pct": np.nan,
            "brown_lesion_pct": np.nan,
            "black_patch_pct": np.nan,
            "yellow_discoloration_pct": np.nan,
            "edge_texture_score": np.nan,
            "disease_risk_score_0_5": np.nan,
            "disease_pattern_flag": "NO_CROP"
        }

    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

    h, s, v = cv2.split(hsv)
    total_pixels = crop_bgr.shape[0] * crop_bgr.shape[1]

    # Dark/black patches: possible rot, deep shadow, or necrotic spots
    black_patch_mask = (v < 45)

    # Brown lesion-like areas
    brown_lesion_mask = (
        (h >= 5) & (h <= 25) &
        (s > 45) &
        (v > 35) & (v < 180)
    )

    # Yellowish discoloration
    yellow_discoloration_mask = (
        (h >= 18) & (h <= 38) &
        (s > 50) &
        (v > 70)
    )

    # Dark spot includes very dark or brown lesion-like area
    dark_spot_mask = black_patch_mask | brown_lesion_mask

    dark_spot_pct = dark_spot_mask.sum() / max(total_pixels, 1) * 100
    brown_lesion_pct = brown_lesion_mask.sum() / max(total_pixels, 1) * 100
    black_patch_pct = black_patch_mask.sum() / max(total_pixels, 1) * 100
    yellow_discoloration_pct = yellow_discoloration_mask.sum() / max(total_pixels, 1) * 100

    # Texture/edge irregularity: proxy for cracking, surface roughness, damage
    edges = cv2.Canny(gray, 80, 160)
    edge_texture_score = edges.sum() / 255 / max(total_pixels, 1) * 100

    # Disease/damage risk scoring, 0 to 5
    risk = 0

    if dark_spot_pct > 2:
        risk += 1
    if dark_spot_pct > 6:
        risk += 1
    if brown_lesion_pct > 4:
        risk += 1
    if black_patch_pct > 5:
        risk += 1
    if edge_texture_score > 12:
        risk += 1

    risk = int(min(risk, 5))

    if risk == 0:
        flag = "LOW_VISIBLE_DISEASE_RISK"
    elif risk <= 2:
        flag = "MILD_VISIBLE_SYMPTOMS"
    elif risk <= 4:
        flag = "MODERATE_VISIBLE_SYMPTOMS"
    else:
        flag = "HIGH_VISIBLE_DISEASE_RISK"

    return {
        "dark_spot_pct": float(dark_spot_pct),
        "brown_lesion_pct": float(brown_lesion_pct),
        "black_patch_pct": float(black_patch_pct),
        "yellow_discoloration_pct": float(yellow_discoloration_pct),
        "edge_texture_score": float(edge_texture_score),
        "disease_risk_score_0_5": risk,
        "disease_pattern_flag": flag
    }


disease_rows = []

for _, row in tqdm(fruit_predictions_df.iterrows(), total=len(fruit_predictions_df), desc="Estimating disease patterns"):

    img_path = Path(row["image_path"])
    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        continue

    x1 = int(row["bbox_x1"])
    y1 = int(row["bbox_y1"])
    x2 = int(row["bbox_x2"])
    y2 = int(row["bbox_y2"])

    crop_bgr = image_bgr[y1:y2, x1:x2]

    disease_features = calculate_disease_pattern_features(crop_bgr)

    disease_rows.append({
        "image_name": row["image_name"],
        "image_path": row["image_path"],
        "detection_id": row["detection_id"],
        "predicted_maturity_class": row["predicted_maturity_class"],
        "confidence": row["confidence"],

        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_x2": x2,
        "bbox_y2": y2,

        "dark_spot_pct": disease_features["dark_spot_pct"],
        "brown_lesion_pct": disease_features["brown_lesion_pct"],
        "black_patch_pct": disease_features["black_patch_pct"],
        "yellow_discoloration_pct": disease_features["yellow_discoloration_pct"],
        "edge_texture_score": disease_features["edge_texture_score"],
        "disease_risk_score_0_5": disease_features["disease_risk_score_0_5"],
        "disease_pattern_flag": disease_features["disease_pattern_flag"],

        "fruit_exposure_score": row.get("fruit_exposure_score", np.nan),
        "discoloration_score": row.get("discoloration_score", np.nan),
        "image_quality_flag": row.get("image_quality_flag", None)
    })

disease_pattern_df = pd.DataFrame(disease_rows)

disease_pattern_path = OUTPUT_DIR / "fruit_level_disease_pattern_estimation.csv"
disease_pattern_df.to_csv(disease_pattern_path, index=False)

print("Saved:", disease_pattern_path)
display(disease_pattern_df.head())

Estimating disease patterns: 100%|██████████| 19953/19953 [00:25<00:00, 786.63it/s]


Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/fruit_level_disease_pattern_estimation.csv


,image_name,image_path,detection_id,predicted_maturity_class,confidence,bbox_x1,bbox_y1,bbox_x2,bbox_y2,dark_spot_pct,brown_lesion_pct,black_patch_pct,yellow_discoloration_pct,edge_texture_score,disease_risk_score_0_5,disease_pattern_flag,fruit_exposure_score,discoloration_score,image_quality_flag
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,0,Green,0.930011,321,351,416,442,27.888953,2.695200,25.251591,50.375940,20.786582,4,MODERATE_VISIBLE_SYMPTOMS,5,4,OK
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,0,Half,0.975460,412,231,560,409,74.218038,60.742484,15.662010,63.741269,5.242180,4,MODERATE_VISIBLE_SYMPTOMS,5,5,OK
2,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,Half,0.966724,159,170,290,328,96.521403,79.650208,19.712049,19.678230,3.140400,4,MODERATE_VISIBLE_SYMPTOMS,5,5,OK
3,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,2,Half,0.906510,267,175,424,339,58.936616,46.496815,15.558490,65.542955,4.582880,4,MODERATE_VISIBLE_SYMPTOMS,5,5,OK
4,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,3,Half,0.901229,124,369,192,446,94.442322,68.468296,32.066463,31.703591,0.000000,4,MODERATE_VISIBLE_SYMPTOMS,5,5,OK


In [16]:
# ==========================================================
# # Cell 14 - Composite Visible Damage / Disease-Risk Index Creation
# ==========================================================
# This creates:
# 1. Fruit-level Composite Disease Index
# 2. Image-level Composite Disease Index
# 3. Disease loss factor for production adjustment
# ==========================================================

import pandas as pd
import numpy as np

# ----------------------------------------------------------
# 1. Helper function to normalize values to 0-100
# ----------------------------------------------------------

def normalize_to_100(value, max_reasonable_value):
    """
    Converts raw feature value to 0-100 scale.
    Caps at 100 to avoid extreme outlier impact.
    """
    if pd.isna(value):
        return 0
    return min((value / max_reasonable_value) * 100, 100)


# ----------------------------------------------------------
# 2. Composite Disease Index at fruit level
# ----------------------------------------------------------
# Weights can be adjusted after expert validation.
# Brown/black/dark patches are given higher weight because they are
# more likely to affect marketable quality.

disease_index_rows = []

for _, row in disease_pattern_df.iterrows():

    dark_spot_index = normalize_to_100(row.get("dark_spot_pct", 0), 20)
    brown_lesion_index = normalize_to_100(row.get("brown_lesion_pct", 0), 15)
    black_patch_index = normalize_to_100(row.get("black_patch_pct", 0), 12)
    yellow_discoloration_index = normalize_to_100(row.get("yellow_discoloration_pct", 0), 20)
    texture_damage_index = normalize_to_100(row.get("edge_texture_score", 0), 25)

    # Convert disease risk score 0-5 into 0-100
    disease_risk_index = normalize_to_100(row.get("disease_risk_score_0_5", 0), 5)

    # Composite weighted disease index
    composite_disease_index = (
        dark_spot_index * 0.20
        + brown_lesion_index * 0.25
        + black_patch_index * 0.20
        + yellow_discoloration_index * 0.10
        + texture_damage_index * 0.10
        + disease_risk_index * 0.15
    )

    composite_disease_index = min(max(composite_disease_index, 0), 100)

    if composite_disease_index < 15:
        disease_severity_band = "Low"
    elif composite_disease_index < 35:
        disease_severity_band = "Mild"
    elif composite_disease_index < 60:
        disease_severity_band = "Moderate"
    else:
        disease_severity_band = "High"

    disease_index_rows.append({
        "image_name": row["image_name"],
        "image_path": row["image_path"],
        "detection_id": row["detection_id"],
        "predicted_maturity_class": row["predicted_maturity_class"],
        "confidence": row["confidence"],

        "dark_spot_index_0_100": dark_spot_index,
        "brown_lesion_index_0_100": brown_lesion_index,
        "black_patch_index_0_100": black_patch_index,
        "yellow_discoloration_index_0_100": yellow_discoloration_index,
        "texture_damage_index_0_100": texture_damage_index,
        "disease_risk_index_0_100": disease_risk_index,

        "composite_disease_index_0_100": composite_disease_index,
        "disease_severity_band": disease_severity_band,

        "dark_spot_pct": row.get("dark_spot_pct", np.nan),
        "brown_lesion_pct": row.get("brown_lesion_pct", np.nan),
        "black_patch_pct": row.get("black_patch_pct", np.nan),
        "yellow_discoloration_pct": row.get("yellow_discoloration_pct", np.nan),
        "edge_texture_score": row.get("edge_texture_score", np.nan),
        "disease_pattern_flag": row.get("disease_pattern_flag", None)
    })

fruit_disease_index_df = pd.DataFrame(disease_index_rows)

fruit_disease_index_path = OUTPUT_DIR / "fruit_level_composite_disease_index.csv"
fruit_disease_index_df.to_csv(fruit_disease_index_path, index=False)

print("Saved:", fruit_disease_index_path)
display(fruit_disease_index_df.head())

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/fruit_level_composite_disease_index.csv


,image_name,image_path,detection_id,predicted_maturity_class,confidence,dark_spot_index_0_100,brown_lesion_index_0_100,black_patch_index_0_100,yellow_discoloration_index_0_100,texture_damage_index_0_100,disease_risk_index_0_100,composite_disease_index_0_100,disease_severity_band,dark_spot_pct,brown_lesion_pct,black_patch_pct,yellow_discoloration_pct,edge_texture_score,disease_pattern_flag
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,0,Green,0.930011,100.0,17.967997,100.0,100.000000,83.146327,80.0,74.806632,High,27.888953,2.695200,25.251591,50.375940,20.786582,MODERATE_VISIBLE_SYMPTOMS
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,0,Half,0.975460,100.0,100.000000,100.0,100.000000,20.968722,80.0,89.096872,High,74.218038,60.742484,15.662010,63.741269,5.242180,MODERATE_VISIBLE_SYMPTOMS
2,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,1,Half,0.966724,100.0,100.000000,100.0,98.391149,12.561600,80.0,88.095275,High,96.521403,79.650208,19.712049,19.678230,3.140400,MODERATE_VISIBLE_SYMPTOMS
3,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,2,Half,0.906510,100.0,100.000000,100.0,100.000000,18.331521,80.0,88.833152,High,58.936616,46.496815,15.558490,65.542955,4.582880,MODERATE_VISIBLE_SYMPTOMS
4,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,3,Half,0.901229,100.0,100.000000,100.0,100.000000,0.000000,80.0,87.000000,High,94.442322,68.468296,32.066463,31.703591,0.000000,MODERATE_VISIBLE_SYMPTOMS


In [17]:
# ==========================================================
# Cell 15 - Image-Level Composite Disease Index
# ==========================================================

image_disease_index_df = (
    fruit_disease_index_df
    .groupby("image_name")
    .agg(
        detected_fruits_for_disease=("detection_id", "count"),
        avg_composite_disease_index=("composite_disease_index_0_100", "mean"),
        max_composite_disease_index=("composite_disease_index_0_100", "max"),
        median_composite_disease_index=("composite_disease_index_0_100", "median"),

        low_disease_fruits=("disease_severity_band", lambda x: (x == "Low").sum()),
        mild_disease_fruits=("disease_severity_band", lambda x: (x == "Mild").sum()),
        moderate_disease_fruits=("disease_severity_band", lambda x: (x == "Moderate").sum()),
        high_disease_fruits=("disease_severity_band", lambda x: (x == "High").sum()),

        avg_dark_spot_pct=("dark_spot_pct", "mean"),
        avg_brown_lesion_pct=("brown_lesion_pct", "mean"),
        avg_black_patch_pct=("black_patch_pct", "mean"),
        avg_yellow_discoloration_pct=("yellow_discoloration_pct", "mean"),
        avg_edge_texture_score=("edge_texture_score", "mean")
    )
    .reset_index()
)

image_disease_index_df["moderate_or_high_disease_fruits"] = (
    image_disease_index_df["moderate_disease_fruits"]
    + image_disease_index_df["high_disease_fruits"]
)

image_disease_index_df["moderate_or_high_disease_pct"] = (
    image_disease_index_df["moderate_or_high_disease_fruits"]
    / image_disease_index_df["detected_fruits_for_disease"]
    * 100
)

image_disease_index_df["high_disease_pct"] = (
    image_disease_index_df["high_disease_fruits"]
    / image_disease_index_df["detected_fruits_for_disease"]
    * 100
)

# ----------------------------------------------------------
# Visible damage / disease-risk loss factor
# This converts the proxy index into expected marketable quality loss.
# ----------------------------------------------------------

def disease_loss_factor_from_index(avg_cdi, moderate_high_pct):
    """
    Returns disease loss factor between 0 and 0.40.
    Example: 0.12 means 12% production loss due to visible disease/damage.
    """

    if pd.isna(avg_cdi):
        return 0.00

    # Base loss from average disease index
    if avg_cdi < 15:
        loss = 0.02
    elif avg_cdi < 35:
        loss = 0.06
    elif avg_cdi < 60:
        loss = 0.15
    else:
        loss = 0.30

    # Additional penalty if many fruits are moderate/high risk
    if pd.notna(moderate_high_pct):
        if moderate_high_pct >= 40:
            loss += 0.10
        elif moderate_high_pct >= 25:
            loss += 0.06
        elif moderate_high_pct >= 10:
            loss += 0.03

    return min(max(loss, 0), 0.40)


image_disease_index_df["disease_loss_factor"] = image_disease_index_df.apply(
    lambda r: disease_loss_factor_from_index(
        r["avg_composite_disease_index"],
        r["moderate_or_high_disease_pct"]
    ),
    axis=1
)

image_disease_index_df["disease_loss_pct"] = (
    image_disease_index_df["disease_loss_factor"] * 100
)

image_disease_index_path = OUTPUT_DIR / "image_level_composite_disease_index.csv"
image_disease_index_df.to_csv(image_disease_index_path, index=False)

print("Saved:", image_disease_index_path)
display(image_disease_index_df.head())

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/image_level_composite_disease_index.csv


,image_name,detected_fruits_for_disease,avg_composite_disease_index,max_composite_disease_index,median_composite_disease_index,low_disease_fruits,mild_disease_fruits,moderate_disease_fruits,high_disease_fruits,avg_dark_spot_pct,avg_brown_lesion_pct,avg_black_patch_pct,avg_yellow_discoloration_pct,avg_edge_texture_score,moderate_or_high_disease_fruits,moderate_or_high_disease_pct,high_disease_pct,disease_loss_factor,disease_loss_pct
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,1,74.806632,74.806632,74.806632,0,0,0,1,27.888953,2.695200,25.251591,50.375940,20.786582,1,100.0,100.0,0.4,40.0
1,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,15,80.692384,89.500000,78.589707,0,0,0,15,55.863690,41.867408,16.162248,48.682496,3.779976,15,100.0,100.0,0.4,40.0
2,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,1,64.686070,64.686070,64.686070,0,0,0,1,92.417980,6.527168,87.716352,2.092483,1.903040,1,100.0,100.0,0.4,40.0
3,-102-_jpg.rf.5af9c7ede77bee134dcb5811dd2efe44.jpg,1,78.100949,78.100949,78.100949,0,0,0,1,97.809229,22.512779,83.092812,0.785896,1.770004,1,100.0,100.0,0.4,40.0
4,-104-_jpg.rf.243a3ea518ad9685625bfe5d9dd0ea40.jpg,1,77.498822,77.498822,77.498822,0,0,0,1,58.420139,10.156250,50.434028,10.543775,8.249628,1,100.0,100.0,0.4,40.0


In [18]:
# ==========================================================
# Cell 16 - Image-Level Future Marketable Production Estimate
# India-focused, image-based production estimation
# Disease-adjusted using Composite Disease Index
# ==========================================================

import pandas as pd
import numpy as np

# ----------------------------------------------------------
# 1. India-focused scenario assumptions
# ----------------------------------------------------------
# Average mature litchi fruit weight varies by cultivar.
# Conservative: smaller fruit / lower size realization
# Base: common good-quality commercial fruit
# Optimistic: larger cultivar / better orchard management

INDIA_PRODUCTION_SCENARIOS = {
    "Conservative": {
        "avg_mature_fruit_weight_g": 18,
        "base_marketable_quality_factor": 0.70,
        "season_flowering_intensity_index": 0.75
    },
    "Base": {
        "avg_mature_fruit_weight_g": 22,
        "base_marketable_quality_factor": 0.80,
        "season_flowering_intensity_index": 0.85
    },
    "Optimistic": {
        "avg_mature_fruit_weight_g": 26,
        "base_marketable_quality_factor": 0.90,
        "season_flowering_intensity_index": 0.95
    }
}

# ----------------------------------------------------------
# 2. Maturity-stage retention assumptions
# ----------------------------------------------------------
# These represent probability that a visible fruit at current stage
# survives and becomes marketable harvest.
#
# Red is already near harvest, so retention is highest.
# Half is close to harvest.
# Green has moderate-to-high retention.
# Young has lower retention due to fruit drop, pest, cracking, weather risk.

STAGE_RETENTION_FACTOR = {
    "Young": 0.70,
    "Green": 0.82,
    "Half": 0.92,
    "Red": 0.98
}

# ----------------------------------------------------------
# 3. Model and visibility correction factors
# ----------------------------------------------------------
# Recall correction: adjusts for fruits missed by the model.
# Occlusion correction: adjusts for fruits hidden behind leaves/branches.
# Image sample scale: if one image is only part of one tree, use >1.
# Example: if one image covers roughly 25% of one tree canopy, use 4.
# If you do not know, keep as 1.0 and call it visible-image estimate.

RECALL_CORRECTION_FACTOR = 1.10
OCCLUSION_CORRECTION_FACTOR = 1.25
IMAGE_SAMPLE_SCALE_FACTOR = 1.00

# ----------------------------------------------------------
# 4. Quality factor adjustment from image-level signals
# ----------------------------------------------------------
# This uses your extracted image features:
# visible_stress_score, image_quality_flag, avg_detection_confidence.
#
# The quality factor starts from scenario base quality.
# Then it is reduced if stress/image quality/confidence is poor.

def calculate_image_quality_factor(
    base_quality_factor,
    visible_stress_score=None,
    image_quality_flag=None,
    avg_detection_confidence=None
):
    quality_factor = base_quality_factor

    # Stress penalty
    if pd.notna(visible_stress_score):
        if visible_stress_score >= 4:
            quality_factor -= 0.15
        elif visible_stress_score >= 3:
            quality_factor -= 0.10
        elif visible_stress_score >= 2:
            quality_factor -= 0.05

    # Image quality penalty
    if isinstance(image_quality_flag, str):
        if image_quality_flag != "OK":
            quality_factor -= 0.05

    # Detection confidence penalty
    if pd.notna(avg_detection_confidence):
        if avg_detection_confidence < 0.40:
            quality_factor -= 0.15
        elif avg_detection_confidence < 0.60:
            quality_factor -= 0.08

    # Keep factor within practical range
    quality_factor = max(0.40, min(quality_factor, 0.95))
    return quality_factor


# ----------------------------------------------------------
# 5. Optional flowering factor
# ----------------------------------------------------------
# Important:
# Since your current images are fruit-stage images, the detected fruit count
# already reflects flowering + fruit set to some extent.
#
# So by default, we keep flowering factor from scenario as a seasonal adjustment.
# If you have reliable image-level flowering_score, you can enable it.
#
# For fruit-stage dataset, recommended: USE_IMAGE_FLOWERING_SCORE = False

USE_IMAGE_FLOWERING_SCORE = False

def flowering_score_to_index(score):
    mapping = {
        0: 1.00,  # At fruit stage, no visible flowers should not reduce production
        1: 0.20,
        2: 0.40,
        3: 0.65,
        4: 0.85,
        5: 1.00
    }
    return mapping.get(int(score), 1.00)


# ----------------------------------------------------------
# 6. Merge image-level disease index into image-level predictions
# ----------------------------------------------------------
# IMPORTANT:
# image_disease_index_df should be created before this cell.
# It should come from:
# Cell 10F - Image-Level Composite Disease Index

required_disease_columns = [
    "image_name",
    "avg_composite_disease_index",
    "moderate_or_high_disease_pct",
    "high_disease_pct",
    "disease_loss_factor",
    "disease_loss_pct"
]

if "image_disease_index_df" in globals():

    missing_cols = [
        col for col in required_disease_columns
        if col not in image_disease_index_df.columns
    ]

    if len(missing_cols) > 0:
        raise ValueError(
            f"image_disease_index_df is missing required columns: {missing_cols}"
        )

    image_predictions_with_disease_df = image_predictions_df.merge(
        image_disease_index_df[required_disease_columns],
        on="image_name",
        how="left"
    )

else:
    print(
        "WARNING: image_disease_index_df not found. "
        "Disease loss will be assumed as zero for all images."
    )

    image_predictions_with_disease_df = image_predictions_df.copy()
    image_predictions_with_disease_df["avg_composite_disease_index"] = 0
    image_predictions_with_disease_df["moderate_or_high_disease_pct"] = 0
    image_predictions_with_disease_df["high_disease_pct"] = 0
    image_predictions_with_disease_df["disease_loss_factor"] = 0
    image_predictions_with_disease_df["disease_loss_pct"] = 0


# If disease index is missing for any image, assume zero disease loss
image_predictions_with_disease_df["avg_composite_disease_index"] = (
    image_predictions_with_disease_df["avg_composite_disease_index"].fillna(0)
)

image_predictions_with_disease_df["moderate_or_high_disease_pct"] = (
    image_predictions_with_disease_df["moderate_or_high_disease_pct"].fillna(0)
)

image_predictions_with_disease_df["high_disease_pct"] = (
    image_predictions_with_disease_df["high_disease_pct"].fillna(0)
)

image_predictions_with_disease_df["disease_loss_factor"] = (
    image_predictions_with_disease_df["disease_loss_factor"].fillna(0)
)

image_predictions_with_disease_df["disease_loss_pct"] = (
    image_predictions_with_disease_df["disease_loss_pct"].fillna(0)
)


# ----------------------------------------------------------
# 7. Main image-level production calculation
# ----------------------------------------------------------

# Carry tree/drone mapping metadata into production output
if "TREE_METADATA_COLUMNS" in globals():
    production_metadata_columns = [
        col for col in ["image_path"] + TREE_METADATA_COLUMNS
        if col in image_predictions_with_disease_df.columns
    ]
else:
    production_metadata_columns = [
        col for col in [
            "image_path",
            "mission_id",
            "flight_id",
            "drone_id",
            "orthomosaic_id",
            "orchard_id",
            "block_id",
            "row_id",
            "tree_id",
            "tree_global_id",
            "tree_group_id",
            "tree_latitude",
            "tree_longitude",
            "tree_centroid_latitude",
            "tree_centroid_longitude",
            "orthomosaic_pixel_x",
            "orthomosaic_pixel_y",
            "local_x_m",
            "local_y_m",
            "gps_match_distance_m",
            "gps_match_status",
            "gps_source",
            "mapping_method",
            "is_simulated_gps"
        ]
        if col in image_predictions_with_disease_df.columns
    ]

production_rows = []

for _, row in image_predictions_with_disease_df.iterrows():

    image_name = row["image_name"]
    production_tree_meta = {
        col: row.get(col, np.nan)
        for col in production_metadata_columns
    }

    young_count = row.get("young_count", 0)
    green_count = row.get("green_count", 0)
    half_count = row.get("half_count", 0)
    red_count = row.get("red_count", 0)

    total_detected_fruit = (
        young_count + green_count + half_count + red_count
    )

    # Stage-adjusted expected harvestable fruit count
    expected_harvestable_fruit_count = (
        young_count * STAGE_RETENTION_FACTOR["Young"]
        + green_count * STAGE_RETENTION_FACTOR["Green"]
        + half_count * STAGE_RETENTION_FACTOR["Half"]
        + red_count * STAGE_RETENTION_FACTOR["Red"]
    )

    # Disease loss factor from Composite Disease Index
    # Example: 0.15 means 15% expected loss due to visible disease/damage
    disease_loss_factor = row.get("disease_loss_factor", 0)

    for scenario_name, scenario in INDIA_PRODUCTION_SCENARIOS.items():

        mature_fruit_weight_g = scenario["avg_mature_fruit_weight_g"]
        base_quality_factor = scenario["base_marketable_quality_factor"]

        if USE_IMAGE_FLOWERING_SCORE:
            flowering_intensity_index = flowering_score_to_index(
                row.get("flowering_score", 0)
            )
        else:
            flowering_intensity_index = scenario["season_flowering_intensity_index"]

        image_quality_factor = calculate_image_quality_factor(
            base_quality_factor=base_quality_factor,
            visible_stress_score=row.get("visible_stress_score", np.nan),
            image_quality_flag=row.get("image_quality_flag", None),
            avg_detection_confidence=row.get("avg_detection_confidence", np.nan)
        )

        # --------------------------------------------------
        # Gross future production before quality factor
        # --------------------------------------------------
        expected_gross_production_kg = (
            expected_harvestable_fruit_count
            * flowering_intensity_index
            * mature_fruit_weight_g
            / 1000
        )

        # --------------------------------------------------
        # Marketable future production before disease loss
        # --------------------------------------------------
        expected_marketable_production_kg_before_disease_loss = (
            expected_gross_production_kg
            * image_quality_factor
            * RECALL_CORRECTION_FACTOR
            * OCCLUSION_CORRECTION_FACTOR
            * IMAGE_SAMPLE_SCALE_FACTOR
        )

        # --------------------------------------------------
        # Disease-loss adjustment for future marketable production
        # --------------------------------------------------
        disease_loss_kg = (
            expected_marketable_production_kg_before_disease_loss
            * disease_loss_factor
        )

        disease_adjusted_marketable_future_production_kg = (
            expected_marketable_production_kg_before_disease_loss
            - disease_loss_kg
        )

        # --------------------------------------------------
        # Harvest-ready current production: mostly Red + Half
        # --------------------------------------------------
        current_harvest_ready_fruit_count = (
            half_count * STAGE_RETENTION_FACTOR["Half"]
            + red_count * STAGE_RETENTION_FACTOR["Red"]
        )

        current_harvest_ready_kg_before_disease_loss = (
            current_harvest_ready_fruit_count
            * mature_fruit_weight_g
            / 1000
            * image_quality_factor
            * RECALL_CORRECTION_FACTOR
            * OCCLUSION_CORRECTION_FACTOR
            * IMAGE_SAMPLE_SCALE_FACTOR
        )

        # --------------------------------------------------
        # Disease-loss adjustment for harvest-ready production
        # --------------------------------------------------
        current_harvest_ready_disease_loss_kg = (
            current_harvest_ready_kg_before_disease_loss
            * disease_loss_factor
        )

        disease_adjusted_current_harvest_ready_kg = (
            current_harvest_ready_kg_before_disease_loss
            - current_harvest_ready_disease_loss_kg
        )

        production_rows.append({
            "image_name": image_name,
            **production_tree_meta,
            "scenario": scenario_name,

            # Fruit counts
            "detected_fruit_count": total_detected_fruit,
            "young_count": young_count,
            "green_count": green_count,
            "half_count": half_count,
            "red_count": red_count,

            # Retention adjusted fruit count
            "expected_harvestable_fruit_count": expected_harvestable_fruit_count,
            "current_harvest_ready_fruit_count": current_harvest_ready_fruit_count,

            # Scenario and production factors
            "flowering_intensity_index": flowering_intensity_index,
            "young_retention_factor": STAGE_RETENTION_FACTOR["Young"],
            "green_retention_factor": STAGE_RETENTION_FACTOR["Green"],
            "half_retention_factor": STAGE_RETENTION_FACTOR["Half"],
            "red_retention_factor": STAGE_RETENTION_FACTOR["Red"],
            "avg_mature_fruit_weight_g": mature_fruit_weight_g,
            "base_marketable_quality_factor": base_quality_factor,
            "image_adjusted_marketable_quality_factor": image_quality_factor,
            "recall_correction_factor": RECALL_CORRECTION_FACTOR,
            "occlusion_correction_factor": OCCLUSION_CORRECTION_FACTOR,
            "image_sample_scale_factor": IMAGE_SAMPLE_SCALE_FACTOR,

            # Disease index and disease-loss factors
            "avg_composite_disease_index": row.get("avg_composite_disease_index", 0),
            "moderate_or_high_disease_pct": row.get("moderate_or_high_disease_pct", 0),
            "high_disease_pct": row.get("high_disease_pct", 0),
            "disease_loss_factor": disease_loss_factor,
            "disease_loss_pct": disease_loss_factor * 100,

            # Production estimates before disease loss
            "expected_gross_future_production_kg": expected_gross_production_kg,
            "expected_marketable_future_production_kg_before_disease_loss": (
                expected_marketable_production_kg_before_disease_loss
            ),
            "current_harvest_ready_estimated_kg_before_disease_loss": (
                current_harvest_ready_kg_before_disease_loss
            ),

            # Disease loss estimates
            "future_production_disease_loss_kg": disease_loss_kg,
            "current_harvest_ready_disease_loss_kg": current_harvest_ready_disease_loss_kg,

            # Final disease-adjusted production estimates
            "disease_adjusted_marketable_future_production_kg": (
                disease_adjusted_marketable_future_production_kg
            ),
            "disease_adjusted_current_harvest_ready_kg": (
                disease_adjusted_current_harvest_ready_kg
            ),

            # Supporting image-level indicators
            "avg_maturity_index_0_100": row.get("avg_maturity_index_0_100", np.nan),
            "avg_detection_confidence": row.get("avg_detection_confidence", np.nan),
            "visible_stress_score": row.get("visible_stress_score", np.nan),
            "canopy_gap_pct": row.get("canopy_gap_pct", np.nan),
            "image_quality_flag": row.get("image_quality_flag", None)
        })

production_estimation_df = pd.DataFrame(production_rows)

production_estimation_path = (
    OUTPUT_DIR / "disease_adjusted_image_level_future_marketable_production_estimation.csv"
)

production_estimation_df.to_csv(production_estimation_path, index=False)

print("Saved:", production_estimation_path)
print("Rows created:", len(production_estimation_df))
print("Expected rows = number of images × 3 scenarios")
display(production_estimation_df.head(12))

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/disease_adjusted_image_level_future_marketable_production_estimation.csv
Rows created: 3924
Expected rows = number of images × 3 scenarios


,image_name,image_path,mission_id,flight_id,drone_id,orthomosaic_id,orchard_id,block_id,row_id,tree_id,...,current_harvest_ready_estimated_kg_before_disease_loss,future_production_disease_loss_kg,current_harvest_ready_disease_loss_kg,disease_adjusted_marketable_future_production_kg,disease_adjusted_current_harvest_ready_kg,avg_maturity_index_0_100,avg_detection_confidence,visible_stress_score,canopy_gap_pct,image_quality_flag
0,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,...,0.000000,0.003349,0.000000,0.005023,0.000000,35.0,0.930011,5,26.984131,OK
1,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,...,0.000000,0.005482,0.000000,0.008223,0.000000,35.0,0.930011,5,26.984131,OK
2,-1-_jpg.rf.acf0acc89ce8b077384fb4d7773d0982.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,...,0.000000,0.008355,0.000000,0.012532,0.000000,35.0,0.930011,5,26.984131,OK
3,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,0.125235,0.054314,0.050094,0.081471,0.075141,55.0,0.694898,5,25.551758,OK
4,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,0.180895,0.088914,0.072358,0.133371,0.108537,55.0,0.694898,5,25.551758,OK
5,-10-_jpg.rf.ebe6973d4b33853fd3d3f8596da0eb98.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,...,0.246675,0.135510,0.098670,0.203266,0.148005,55.0,0.694898,5,25.551758,OK
6,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T003,...,0.012524,0.003757,0.005009,0.005636,0.007514,65.0,0.801126,5,26.724854,OK
7,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T003,...,0.018090,0.006150,0.007236,0.009226,0.010854,65.0,0.801126,5,26.724854,OK
8,-100-_jpg.rf.c05b3c16382da7f2d4f22f55c59da5bc.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T003,...,0.024668,0.009374,0.009867,0.014060,0.014801,65.0,0.801126,5,26.724854,OK
9,-102-_jpg.rf.5af9c7ede77bee134dcb5811dd2efe44.jpg,/Users/utkarsh/Desktop/Projects/IridAxis Globa...,MISSION_001,FLIGHT_ORCHARD_01,DRONE_001,ORTHO_ORCHARD_01,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T004,...,0.012524,0.003757,0.005009,0.005636,0.007514,65.0,0.969044,5,25.729004,OK


In [19]:
# Cell 17 — Tree-level disease-adjusted production summary

tree_production_group_columns = [
    "scenario",
    "orchard_id",
    "block_id",
    "row_id",
    "tree_id",
    "tree_global_id",
    "mission_id",
    "flight_id",
    "orthomosaic_id"
]

tree_production_geo_columns = [
    "tree_latitude",
    "tree_longitude",
    "tree_centroid_latitude",
    "tree_centroid_longitude",
    "orthomosaic_pixel_x",
    "orthomosaic_pixel_y",
    "local_x_m",
    "local_y_m",
    "gps_match_distance_m",
    "gps_match_status",
    "gps_source",
    "mapping_method",
    "is_simulated_gps"
]

tree_production_group_columns = [
    col for col in tree_production_group_columns
    if col in production_estimation_df.columns
]

tree_production_geo_columns = [
    col for col in tree_production_geo_columns
    if col in production_estimation_df.columns
]

tree_production_agg_dict = {
    "image_name": "nunique",
    "detected_fruit_count": "sum",
    "young_count": "sum",
    "green_count": "sum",
    "half_count": "sum",
    "red_count": "sum",
    "expected_harvestable_fruit_count": "sum",
    "current_harvest_ready_fruit_count": "sum",
    "expected_marketable_future_production_kg_before_disease_loss": "sum",
    "future_production_disease_loss_kg": "sum",
    "disease_adjusted_marketable_future_production_kg": "sum",
    "current_harvest_ready_estimated_kg_before_disease_loss": "sum",
    "current_harvest_ready_disease_loss_kg": "sum",
    "disease_adjusted_current_harvest_ready_kg": "sum",
    "avg_composite_disease_index": "mean",
    "disease_loss_factor": "mean",
    "disease_loss_pct": "mean",
    "avg_detection_confidence": "mean",
    "visible_stress_score": "mean",
    "canopy_gap_pct": "mean"
}

for col in tree_production_geo_columns:
    tree_production_agg_dict[col] = "first"

tree_level_production_summary_df = (
    production_estimation_df
    .groupby(tree_production_group_columns, dropna=False)
    .agg(tree_production_agg_dict)
    .reset_index()
    .rename(columns={
        "image_name": "total_images",
        "detected_fruit_count": "total_detected_fruits"
    })
)

tree_level_production_summary_path = OUTPUT_DIR / "tree_level_disease_adjusted_production_summary.csv"
tree_level_production_summary_df.to_csv(tree_level_production_summary_path, index=False)

print("Saved:", tree_level_production_summary_path)
print("Tree-level production rows:", len(tree_level_production_summary_df))

display(tree_level_production_summary_df.head())


Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/tree_level_disease_adjusted_production_summary.csv
Tree-level production rows: 3924


,scenario,orchard_id,block_id,row_id,tree_id,tree_global_id,mission_id,flight_id,orthomosaic_id,total_images,...,tree_centroid_longitude,orthomosaic_pixel_x,orthomosaic_pixel_y,local_x_m,local_y_m,gps_match_distance_m,gps_match_status,gps_source,mapping_method,is_simulated_gps
0,Base,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T001,O01_B01_R01_T001,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,...,88.400000,0,0,0,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
1,Base,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T002,O01_B01_R01_T002,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,...,88.400050,100,0,5,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
2,Base,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T003,O01_B01_R01_T003,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,...,88.400101,200,0,10,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
3,Base,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T004,O01_B01_R01_T004,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,...,88.400151,300,0,15,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True
4,Base,ORCHARD_01,BLOCK_01,ROW_01,O01_B01_R01_T005,O01_B01_R01_T005,MISSION_001,FLIGHT_ORCHARD_01,ORTHO_ORCHARD_01,1,...,88.400201,400,0,20,0,2.1,DRONE_MATCHED,SIMULATED_DRONE_MAPPING,DYNAMIC_FROM_IMAGE_FOLDER,True


In [20]:
# ==========================================================
# Cell 18 - Disease-adjusted Production Summary
# Before disease vs after disease comparison
# ==========================================================

production_summary_df = (
    production_estimation_df
    .groupby("scenario")
    .agg(
        total_images=("image_name", "nunique"),
        total_detected_fruits=("detected_fruit_count", "sum"),

        # Future production before disease
        total_future_production_before_disease_kg=(
            "expected_marketable_future_production_kg_before_disease_loss", "sum"
        ),

        # Disease loss
        total_future_disease_loss_kg=(
            "future_production_disease_loss_kg", "sum"
        ),

        # Future production after disease
        total_future_production_after_disease_kg=(
            "disease_adjusted_marketable_future_production_kg", "sum"
        ),

        # Current harvest-ready before disease
        total_current_harvest_ready_before_disease_kg=(
            "current_harvest_ready_estimated_kg_before_disease_loss", "sum"
        ),

        # Current harvest-ready disease loss
        total_current_harvest_ready_disease_loss_kg=(
            "current_harvest_ready_disease_loss_kg", "sum"
        ),

        # Current harvest-ready after disease
        total_current_harvest_ready_after_disease_kg=(
            "disease_adjusted_current_harvest_ready_kg", "sum"
        ),

        avg_disease_loss_factor=("disease_loss_factor", "mean"),
        avg_disease_loss_pct=("disease_loss_pct", "mean"),
        avg_composite_disease_index=("avg_composite_disease_index", "mean"),
        avg_detection_confidence=("avg_detection_confidence", "mean")
    )
    .reset_index()
)

# Calculate overall disease loss percentage for future production
production_summary_df["future_disease_loss_pct_calculated"] = (
    production_summary_df["total_future_disease_loss_kg"]
    / production_summary_df["total_future_production_before_disease_kg"]
    * 100
)

# Convert kg to tonnes
production_summary_df["total_future_production_before_disease_tonnes"] = (
    production_summary_df["total_future_production_before_disease_kg"] / 1000
)

production_summary_df["total_future_disease_loss_tonnes"] = (
    production_summary_df["total_future_disease_loss_kg"] / 1000
)

production_summary_df["total_future_production_after_disease_tonnes"] = (
    production_summary_df["total_future_production_after_disease_kg"] / 1000
)

production_summary_path = OUTPUT_DIR / "disease_loss_production_summary_before_after.csv"
production_summary_df.to_csv(production_summary_path, index=False)

print("Saved:", production_summary_path)
display(production_summary_df)

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/disease_loss_production_summary_before_after.csv


,scenario,total_images,total_detected_fruits,total_future_production_before_disease_kg,total_future_disease_loss_kg,total_future_production_after_disease_kg,total_current_harvest_ready_before_disease_kg,total_current_harvest_ready_disease_loss_kg,total_current_harvest_ready_after_disease_kg,avg_disease_loss_factor,avg_disease_loss_pct,avg_composite_disease_index,avg_detection_confidence,future_disease_loss_pct_calculated,total_future_production_before_disease_tonnes,total_future_disease_loss_tonnes,total_future_production_after_disease_tonnes
0,Base,1308,19953,291.332581,82.312770,209.019811,191.151299,57.861352,133.289947,0.295199,29.519878,56.869909,0.775609,28.253884,0.291333,0.082313,0.209020
1,Conservative,1308,19953,178.220590,50.281738,127.938851,132.504501,40.034960,92.469541,0.295199,29.519878,56.869909,0.775609,28.213204,0.178221,0.050282,0.127939
2,Optimistic,1308,19953,443.538879,125.449610,318.089269,260.416771,78.934920,181.481851,0.295199,29.519878,56.869909,0.775609,28.283791,0.443539,0.125450,0.318089


In [21]:
# ==========================================================
# Cell 19 - Disease Pattern Summary by Maturity Stage
# ==========================================================

disease_stage_summary_df = (
    disease_pattern_df
    .groupby("predicted_maturity_class")
    .agg(
        detected_fruits=("predicted_maturity_class", "count"),
        avg_confidence=("confidence", "mean"),
        avg_dark_spot_pct=("dark_spot_pct", "mean"),
        avg_brown_lesion_pct=("brown_lesion_pct", "mean"),
        avg_black_patch_pct=("black_patch_pct", "mean"),
        avg_yellow_discoloration_pct=("yellow_discoloration_pct", "mean"),
        avg_edge_texture_score=("edge_texture_score", "mean"),
        avg_disease_risk_score=("disease_risk_score_0_5", "mean"),
        high_risk_fruits=(
            "disease_pattern_flag",
            lambda x: (x == "HIGH_VISIBLE_DISEASE_RISK").sum()
        ),
        moderate_symptom_fruits=(
            "disease_pattern_flag",
            lambda x: (x == "MODERATE_VISIBLE_SYMPTOMS").sum()
        ),
        mild_symptom_fruits=(
            "disease_pattern_flag",
            lambda x: (x == "MILD_VISIBLE_SYMPTOMS").sum()
        ),
        low_risk_fruits=(
            "disease_pattern_flag",
            lambda x: (x == "LOW_VISIBLE_DISEASE_RISK").sum()
        )
    )
    .reset_index()
)

disease_stage_summary_df["high_risk_pct"] = (
    disease_stage_summary_df["high_risk_fruits"]
    / disease_stage_summary_df["detected_fruits"]
    * 100
)

disease_stage_summary_df["moderate_or_high_risk_pct"] = (
    (
        disease_stage_summary_df["moderate_symptom_fruits"]
        + disease_stage_summary_df["high_risk_fruits"]
    )
    / disease_stage_summary_df["detected_fruits"]
    * 100
)

disease_stage_summary_path = OUTPUT_DIR / "disease_pattern_summary_by_maturity_stage.csv"
disease_stage_summary_df.to_csv(disease_stage_summary_path, index=False)

print("Saved:", disease_stage_summary_path)
display(disease_stage_summary_df)

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/disease_pattern_summary_by_maturity_stage.csv


,predicted_maturity_class,detected_fruits,avg_confidence,avg_dark_spot_pct,avg_brown_lesion_pct,avg_black_patch_pct,avg_yellow_discoloration_pct,avg_edge_texture_score,avg_disease_risk_score,high_risk_fruits,moderate_symptom_fruits,mild_symptom_fruits,low_risk_fruits,high_risk_pct,moderate_or_high_risk_pct
0,Green,5142,0.750963,16.702242,10.624523,6.447318,66.238211,15.497070,2.988331,406,3148,1383,205,7.895760,69.117075
1,Half,1154,0.748333,58.626200,35.400996,25.703417,25.255213,9.132434,3.956672,198,955,1,0,17.157712,99.913345
2,Red,8767,0.681898,18.567360,12.321762,6.763217,6.409259,15.426629,3.491958,1204,6200,1169,194,13.733318,84.453063
3,Young,4890,0.737595,9.065971,4.234414,4.917589,52.387808,21.743160,2.615542,312,2174,2312,92,6.380368,50.838446


In [22]:
# Cell 20 — Manual validation metrics at IoU 0.50
def load_ground_truth_boxes(img_path):
    label_path = SOURCE_LABEL_DIR / f"{img_path.stem}.txt"
    image_bgr = cv2.imread(str(img_path))
    
    if image_bgr is None:
        return []
    
    img_h, img_w = image_bgr.shape[:2]
    labels = read_yolo_label(label_path)
    
    gt_boxes = []
    for lab in labels:
        x1, y1, x2, y2 = yolo_xywh_to_xyxy(
            lab["x_center"],
            lab["y_center"],
            lab["bbox_width"],
            lab["bbox_height"],
            img_w,
            img_h
        )
        gt_boxes.append({
            "class_id": lab["class_id"],
            "box": [x1, y1, x2, y2],
            "matched": False
        })
    
    return gt_boxes


validation_image_dir = WORK_DIR / "images" / "val"
validation_images = []
for ext in image_extensions:
    validation_images.extend(list(validation_image_dir.glob(ext)))

eval_rows = []

for img_path in tqdm(validation_images, desc="Manual IoU validation"):
    source_img_path = SOURCE_IMAGE_DIR / img_path.name
    gt_boxes = load_ground_truth_boxes(source_img_path)   

    results = trained_model.predict(
        source=str(img_path),
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=VAL_DEVICE,
        max_det=YOLO_MAX_DET,
        verbose=False
    )
    pred_boxes = []
    boxes = results[0].boxes
    
    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            
            pred_boxes.append({
                "class_id": cls_id,
                "box": [int(v) for v in xyxy],
                "confidence": conf,
                "matched": False
            })
    
    # Match predictions to GT by class and IoU
    for pred in sorted(pred_boxes, key=lambda x: x["confidence"], reverse=True):
        best_iou = 0
        best_gt_idx = None
        
        for gt_idx, gt in enumerate(gt_boxes):
            if gt["matched"]:
                continue
            if gt["class_id"] != pred["class_id"]:
                continue
            
            iou = calculate_iou(pred["box"], gt["box"])
            
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_gt_idx is not None and best_iou >= IOU_THRESHOLD:
            pred["matched"] = True
            gt_boxes[best_gt_idx]["matched"] = True
            
            eval_rows.append({
                "image_name": img_path.name,
                "class_id": pred["class_id"],
                "class_name": CLASS_NAMES.get(pred["class_id"], "UNKNOWN"),
                "confidence": pred["confidence"],
                "iou": best_iou,
                "result": "TP"
            })
        else:
            eval_rows.append({
                "image_name": img_path.name,
                "class_id": pred["class_id"],
                "class_name": CLASS_NAMES.get(pred["class_id"], "UNKNOWN"),
                "confidence": pred["confidence"],
                "iou": best_iou,
                "result": "FP"
            })
    
    # Unmatched GT are false negatives
    for gt in gt_boxes:
        if not gt["matched"]:
            eval_rows.append({
                "image_name": img_path.name,
                "class_id": gt["class_id"],
                "class_name": CLASS_NAMES.get(gt["class_id"], "UNKNOWN"),
                "confidence": np.nan,
                "iou": np.nan,
                "result": "FN"
            })

manual_eval_df = pd.DataFrame(eval_rows)
manual_eval_path = OUTPUT_DIR / "manual_iou50_detection_validation_rows.csv"
manual_eval_df.to_csv(manual_eval_path, index=False)

summary_rows = []

for cls_id, cls_name in CLASS_NAMES.items():
    cls_df = manual_eval_df[manual_eval_df["class_id"] == cls_id]
    
    tp = (cls_df["result"] == "TP").sum()
    fp = (cls_df["result"] == "FP").sum()
    fn = (cls_df["result"] == "FN").sum()
    
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    
    summary_rows.append({
        "class_id": cls_id,
        "class_name": cls_name,
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "precision_iou50": precision,
        "recall_iou50": recall,
        "f1_iou50": f1
    })

manual_validation_summary_df = pd.DataFrame(summary_rows)
manual_validation_summary_path = OUTPUT_DIR / "manual_iou50_validation_summary_by_class.csv"
manual_validation_summary_df.to_csv(manual_validation_summary_path, index=False)

print("Saved:", manual_eval_path)
print("Saved:", manual_validation_summary_path)

display(manual_validation_summary_df)

Manual IoU validation: 100%|██████████| 262/262 [00:13<00:00, 20.02it/s]

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/manual_iou50_detection_validation_rows.csv
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/manual_iou50_validation_summary_by_class.csv


,class_id,class_name,true_positives,false_positives,false_negatives,precision_iou50,recall_iou50,f1_iou50
0,0,Green,849,262,56,0.764176,0.938122,0.842262
1,1,Half,142,83,45,0.631111,0.759358,0.689320
2,2,Red,1269,518,100,0.710129,0.926954,0.804183
3,3,Young,822,358,19,0.696610,0.977408,0.813459


In [23]:
# Cell 21 — Prediction and estimation statistics
prediction_stats = []

if len(fruit_predictions_df) > 0:
    maturity_summary = (
        fruit_predictions_df
        .groupby("predicted_maturity_class")
        .agg(
            fruit_count=("predicted_maturity_class", "count"),
            avg_confidence=("confidence", "mean"),
            std_confidence=("confidence", "std"),
            min_confidence=("confidence", "min"),
            max_confidence=("confidence", "max"),
            avg_bbox_area_pct=("bbox_area_pct", "mean"),
            avg_exposure_score=("fruit_exposure_score", "mean"),
            avg_discoloration_score=("discoloration_score", "mean"),
            avg_maturity_index=("maturity_index_0_100", "mean")
        )
        .reset_index()
    )
else:
    maturity_summary = pd.DataFrame()

image_summary = pd.DataFrame([{
    "total_images_scored": image_predictions_df.shape[0],
    "total_detected_fruits": image_predictions_df["detected_fruit_count"].sum(),
    "avg_detected_fruits_per_image": image_predictions_df["detected_fruit_count"].mean(),
    "avg_image_confidence": image_predictions_df["avg_detection_confidence"].mean(),
    "avg_flowering_score": image_predictions_df["flowering_score"].mean(),
    "avg_visible_stress_score": image_predictions_df["visible_stress_score"].mean(),
    "avg_canopy_gap_pct": image_predictions_df["canopy_gap_pct"].mean(),
    "avg_green_canopy_pct_proxy": image_predictions_df["green_canopy_pct_proxy"].mean(),
    "avg_thermal_proxy_score": image_predictions_df["thermal_proxy_score"].mean(),
    "avg_blur_score": image_predictions_df["blur_score"].mean(),
    "avg_brightness_mean": image_predictions_df["brightness_mean"].mean(),
    "avg_contrast_std": image_predictions_df["contrast_std"].mean()
}])

maturity_summary_path = OUTPUT_DIR / "prediction_statistics_by_maturity_class.csv"
image_summary_path = OUTPUT_DIR / "image_level_research_summary_statistics.csv"

maturity_summary.to_csv(maturity_summary_path, index=False)
image_summary.to_csv(image_summary_path, index=False)

print("Saved:", maturity_summary_path)
print("Saved:", image_summary_path)

display(maturity_summary)
display(image_summary)

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/prediction_statistics_by_maturity_class.csv
Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/image_level_research_summary_statistics.csv


,predicted_maturity_class,fruit_count,avg_confidence,std_confidence,min_confidence,max_confidence,avg_bbox_area_pct,avg_exposure_score,avg_discoloration_score,avg_maturity_index
0,Green,5142,0.750963,0.174865,0.250726,0.972882,1.285698,4.381369,2.590432,35.0
1,Half,1154,0.748333,0.206490,0.250514,0.986750,2.029419,4.220971,4.761698,65.0
2,Red,8767,0.681898,0.197806,0.250037,0.982350,0.896858,3.898141,3.013346,90.0
3,Young,4890,0.737595,0.180499,0.250480,0.999015,0.859907,4.162986,1.744376,10.0


,total_images_scored,total_detected_fruits,avg_detected_fruits_per_image,avg_image_confidence,avg_flowering_score,avg_visible_stress_score,avg_canopy_gap_pct,avg_green_canopy_pct_proxy,avg_thermal_proxy_score,avg_blur_score,avg_brightness_mean,avg_contrast_std
0,1308,19953,15.254587,0.775609,4.935015,4.256116,20.389626,43.007169,2.146789,794.172225,112.672205,69.687085


In [24]:
# Cell 22-Generate exact observation counts
from pathlib import Path
import pandas as pd
import numpy as np

summary_rows = []

# Original dataset counts
summary_rows.append({
    "stage": "Original Dataset",
    "file_or_object": "Original images",
    "observation_level": "Image",
    "row_count": len(list(SOURCE_IMAGE_DIR.glob("*.jpg"))) 
                 + len(list(SOURCE_IMAGE_DIR.glob("*.jpeg")))
                 + len(list(SOURCE_IMAGE_DIR.glob("*.png")))
})

summary_rows.append({
    "stage": "Original Dataset",
    "file_or_object": "Original YOLO label files",
    "observation_level": "Label file",
    "row_count": len(list(SOURCE_LABEL_DIR.glob("*.txt")))
})

if "label_audit_df" in globals():
    summary_rows.append({
        "stage": "Original Dataset",
        "file_or_object": "Original labelled objects",
        "observation_level": "Fruit object / bounding box",
        "row_count": label_audit_df[label_audit_df["has_label"] == True].shape[0]
    })

# Train/validation split counts
train_img_dir = WORK_DIR / "images" / "train"
val_img_dir = WORK_DIR / "images" / "val"
train_label_dir = WORK_DIR / "labels" / "train"
val_label_dir = WORK_DIR / "labels" / "val"

if train_img_dir.exists():
    summary_rows.append({
        "stage": "After Train/Validation Split",
        "file_or_object": "Training images",
        "observation_level": "Image",
        "row_count": len(list(train_img_dir.glob("*.*")))
    })

if val_img_dir.exists():
    summary_rows.append({
        "stage": "After Train/Validation Split",
        "file_or_object": "Validation images",
        "observation_level": "Image",
        "row_count": len(list(val_img_dir.glob("*.*")))
    })

if train_label_dir.exists():
    summary_rows.append({
        "stage": "After Train/Validation Split",
        "file_or_object": "Training label files",
        "observation_level": "Label file",
        "row_count": len(list(train_label_dir.glob("*.txt")))
    })

if val_label_dir.exists():
    summary_rows.append({
        "stage": "After Train/Validation Split",
        "file_or_object": "Validation label files",
        "observation_level": "Label file",
        "row_count": len(list(val_label_dir.glob("*.txt")))
    })

# Output CSV counts
csv_files = list(OUTPUT_DIR.glob("*.csv"))

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        summary_rows.append({
            "stage": "Generated Output",
            "file_or_object": csv_file.name,
            "observation_level": "CSV rows",
            "row_count": df.shape[0],
            "column_count": df.shape[1]
        })
    except Exception as e:
        summary_rows.append({
            "stage": "Generated Output",
            "file_or_object": csv_file.name,
            "observation_level": "CSV read error",
            "row_count": np.nan,
            "column_count": np.nan
        })

observation_count_summary_df = pd.DataFrame(summary_rows)

observation_count_summary_path = OUTPUT_DIR / "observation_count_summary.csv"
observation_count_summary_df.to_csv(observation_count_summary_path, index=False)

print("Saved:", observation_count_summary_path)
display(observation_count_summary_df)

Saved: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/research_outputs/observation_count_summary.csv


,stage,file_or_object,observation_level,row_count,column_count
0,Original Dataset,Original images,Image,1308,NaN
1,Original Dataset,Original YOLO label files,Label file,1308,NaN
2,Original Dataset,Original labelled objects,Fruit object / bounding box,15759,NaN
3,After Train/Validation Split,Training images,Image,1045,NaN
4,After Train/Validation Split,Validation images,Image,262,NaN
5,After Train/Validation Split,Training label files,Label file,1045,NaN
6,After Train/Validation Split,Validation label files,Label file,262,NaN
7,Generated Output,disease_adjusted_image_level_future_marketable...,CSV rows,3924,82.0
8,Generated Output,disease_pattern_summary_by_maturity_stage.csv,CSV rows,4,15.0
9,Generated Output,tree_level_disease_adjusted_production_summary...,CSV rows,3924,42.0


In [25]:
# Set your working folder
import os

PROJECT_DIR = "/Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis"

os.chdir(PROJECT_DIR)

print("Current folder:", os.getcwd())
print("Files:", os.listdir()[:10])

Current folder: /Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis
Files: ['LICHI_IMAGE_ANALYSIS_PYTHON_DESCRIPTION.xlsx', 'classes.txt', 'image_tree_mapping.csv', '.DS_Store', 'requirements.txt', 'yolo11n.pt', 'Litchi_project', 'info.txt', 'data_sorce_etc.docx', 'OLD']


In [26]:
# Add ngrok auth token
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3DXJXDqDlLtwgec9aMSvp2FZc3A_5v1KULjQXfABT5eG8pGSW"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok auth token configured.")

ngrok auth token configured.


In [27]:
#  Start Streamlit in background
import subprocess
import time
import os
import signal

APP_FILE = "lichi_streamlit_story_dashboard_app_fixed_v2.py"
PORT = 8501

# Stop any previous Streamlit process started from this notebook, if present
try:
    streamlit_process.terminate()
    time.sleep(2)
except NameError:
    pass

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", APP_FILE,
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

print("Streamlit started.")
print("Local URL: http://localhost:8501")

Streamlit started.
Local URL: http://localhost:8501


In [28]:
# Create public ngrok link
from pyngrok import ngrok

# Close old tunnels if any
ngrok.kill()

public_url = ngrok.connect(8501, "http")

print("Public dashboard URL:")
print(public_url)

Public dashboard URL:
NgrokTunnel: "https://flaxseed-zombie-prevail.ngrok-free.dev" -> "http://localhost:8501"


In [29]:

breakpoint

<function breakpoint>

In [31]:
!streamlit run lichi_streamlit_story_dashboard_app_fixed_v2.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.29.114:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
2026-05-11 18:14:30.956 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-11 18:14:30.968 Serialization of dataframe to Arrow table was unsuccessful. Applying automatic fixes for column types to make the dataframe Arrow-compatible.
Traceback (most recent call last):
  File "/Users/utkarsh/Desktop/Projects/IridAxis Global Pvt. Ltd./Innovatio_research/Lichi/Product_Development/Lichi_Image _Analysis/Litchi_project/lib/python3.9/site-packages/streamlit/dataframe_util.py", line 822, in convert_pandas_df_to_arrow_bytes
    table = pa.Table.from_pandas(df)
  File "py

In [ ]:
#  Sharing of link to third party so that he can look the dashbord
# Run Below code from terminal-
